In [1]:
import pandas as pd
import geopandas as gpd
from os import path, environ, makedirs
from dotenv import load_dotenv
from unidecode import unidecode

from core.geo import areal_weighted_interpolation

In [2]:
load_dotenv()

True

# Carregando os dados extraídos no notebook anterior

Neste notebook, vamos utilizar os dados extraídos e salvos pelo notebook `03 habitação - extração.ipynb`.

In [3]:
input_dir = path.join('data', 'cache', 'urbanismo')

In [4]:
filename = path.join(input_dir, 'orcamento_habitacao_original.csv')
df_orcamento = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8',
            dtype=str)
df_orcamento

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Congelado,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao
0,01/01/2024,31/12/2024,2024,2024,169113,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00
1,01/01/2024,31/12/2024,2024,2024,167099,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00
2,01/01/2024,31/12/2024,2024,2024,167101,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18 00:00:00
3,01/01/2024,31/12/2024,2024,2024,171051,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,14985177,14963773.14,14963773.14,14842626.58,14828557.72,"21403,86",2025-01-18 00:00:00
4,01/01/2024,31/12/2024,2024,2024,173960,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,10000000,9968965.45,9968965.45,6827361.22,6827361.22,"31034,55",2025-01-18 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,01/01/2024,31/12/2024,2024,2024,182123,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,2025-01-18 00:00:00
392,01/01/2024,31/12/2024,2024,2024,182125,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,2025-01-18 00:00:00
393,01/01/2024,31/12/2024,2024,2024,164701,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,2745125.68,0.0,2745125.68,"10311885,81",6264151.26,6264151.26,5565823.14,5492338.9,"4047734,55",2025-01-18 00:00:00
394,01/01/2024,31/12/2024,2024,2024,182481,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,2025-01-18 00:00:00


In [5]:
for col in [col for col in df_orcamento.columns if 'Vl' in col]:
    df_orcamento[col] = df_orcamento[col].astype(float)
df_orcamento['DataExtracao'] = pd.to_datetime(df_orcamento['DataExtracao'])
df_orcamento

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Congelado,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao
0,01/01/2024,31/12/2024,2024,2024,169113,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.00,0.0,0.00,1000,0.00,0.00,0.00,0.00,1000,2025-01-18
1,01/01/2024,31/12/2024,2024,2024,167099,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.00,0.0,0.00,1000,0.00,0.00,0.00,0.00,1000,2025-01-18
2,01/01/2024,31/12/2024,2024,2024,167101,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.00,0.0,0.00,1000,0.00,0.00,0.00,0.00,1000,2025-01-18
3,01/01/2024,31/12/2024,2024,2024,171051,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.00,0.0,0.00,14985177,14963773.14,14963773.14,14842626.58,14828557.72,"21403,86",2025-01-18
4,01/01/2024,31/12/2024,2024,2024,173960,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.00,0.0,0.00,10000000,9968965.45,9968965.45,6827361.22,6827361.22,"31034,55",2025-01-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,01/01/2024,31/12/2024,2024,2024,182123,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.00,0.0,0.00,0,0.00,0.00,0.00,0.00,0,2025-01-18
392,01/01/2024,31/12/2024,2024,2024,182125,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.00,0.0,0.00,0,0.00,0.00,0.00,0.00,0,2025-01-18
393,01/01/2024,31/12/2024,2024,2024,164701,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,2745125.68,0.0,2745125.68,"10311885,81",6264151.26,6264151.26,5565823.14,5492338.90,"4047734,55",2025-01-18
394,01/01/2024,31/12/2024,2024,2024,182481,Indireta,91,FMH,Fundo Municipal de Habitação,10,...,0.00,0.0,0.00,0,0.00,0.00,0.00,0.00,0,2025-01-18


In [6]:
filename = path.join(input_dir, 'orcamento_regionalizado_habitacao_original.csv')
df_orcamento_r = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8',
            dtype=str)
df_orcamento_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,DESCRIÇÃO_FONTE,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,01,41426,2024,94288,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,531084.11
1,01,41227,2024,93857,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,231882.59
2,01,41120,2024,122765,2024,2024-05-07 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Norte,Subprefeitura Casa Verde/Cachoeirinha,Supra-Distrital,Despesa Regionalizável,240538.93
3,01,41227,2024,93867,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,294777.93
4,01,41426,2024,94290,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,299606.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6615,01,51136,2024,237458,2024,2024-08-27 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,940786.45
6616,01,51133,2024,166275,2024,2024-06-21 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,1407083.17
6617,01,51133,2024,195419,2024,2024-07-23 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,1936628.4
6618,01,51133,2024,237449,2024,2024-08-27 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,815880.37


In [7]:
df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'] = df_orcamento_r['VALOR_DETALHAMENTO_AÇÃO'].astype(float)
df_orcamento_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,DESCRIÇÃO_FONTE,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,01,41426,2024,94288,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,531084.11
1,01,41227,2024,93857,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,231882.59
2,01,41120,2024,122765,2024,2024-05-07 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Norte,Subprefeitura Casa Verde/Cachoeirinha,Supra-Distrital,Despesa Regionalizável,240538.93
3,01,41227,2024,93867,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,294777.93
4,01,41426,2024,94290,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,299606.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6615,01,51136,2024,237458,2024,2024-08-27 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,940786.45
6616,01,51133,2024,166275,2024,2024-06-21 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,1407083.17
6617,01,51133,2024,195419,2024,2024-07-23 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,1936628.40
6618,01,51133,2024,237449,2024,2024-08-27 00:00:00.0000000,98,FUNDURB,Fundo de Desenvolvimento Urbano,14,...,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,2,759,0402,1,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,815880.37


In [8]:
df_orcamento_r.dtypes

COD_EMPRESA_PMSP                     object
COD_EMPENHO                          object
ANO_EMPENHO                          object
CÓDIGO_NLP                           object
ANO_LIQUIDAÇÃO                       object
DATA_LIQUIDAÇÃO                      object
CÓDIGO_ÓRGÃO                         object
SIGLA_ÓRGÃO                          object
DESCRIÇÃO_ÓRGÃO                      object
CÓDIGO_UNIDADE                       object
DESCRIÇÃO_UNIDADE                    object
CÓDIGO_FUNÇÃO                        object
DESCRIÇÃO_FUNÇÃO                     object
CÓDIGO_SUBFUNÇÃO                     object
DESCRIÇÃO_SUBFUNÇÃO                  object
CÓDIGO_PROGRAMA                      object
DESCRIÇÃO_PROGRAMA                   object
CÓDIGO_PROJ_ATIV                     object
DESCRIÇÃO_PROJ_ATIV                  object
CÓDIGO_CONTA_DESPESA                 object
DESCRIÇÃO_CONTA_DESPESA              object
CÓDIGO_FONTE                         object
DESCRIÇÃO_FONTE                 

In [9]:
filename = path.join(input_dir, 'pdm_meta_12_original.csv')
df_meta_12 = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8')
df_meta_12

,Número da Meta,Descrição,Área,Subprefeitura,2021,2022,2023,2024
0,12,Prover 49.000 moradias de interesse social.,Habitação,Butantã,216.0,546.0,3081.0,7261.0
1,12,Prover 49.000 moradias de interesse social.,Habitação,Campo Limpo,1316.0,1748.0,1748.0,2161.0
2,12,Prover 49.000 moradias de interesse social.,Habitação,Capela do Socorro,562.0,2040.0,2124.0,2140.0
3,12,Prover 49.000 moradias de interesse social.,Habitação,Casa Verde,0.0,108.0,128.0,128.0
4,12,Prover 49.000 moradias de interesse social.,Habitação,Cidade Ademar,0.0,8.0,78.0,78.0
5,12,Prover 49.000 moradias de interesse social.,Habitação,Cidade Tiradentes,50.0,50.0,347.0,507.0
6,12,Prover 49.000 moradias de interesse social.,Habitação,Ermelino Matarazzo,0.0,0.0,0.0,266.0
7,12,Prover 49.000 moradias de interesse social.,Habitação,Freguesia/Brasilândia,228.0,228.0,228.0,502.0
8,12,Prover 49.000 moradias de interesse social.,Habitação,Guaianases,0.0,968.0,1122.0,1322.0
9,12,Prover 49.000 moradias de interesse social.,Habitação,Ipiranga,1694.0,1965.0,2205.0,2685.0


In [10]:
filename = path.join(input_dir, 'his_entregue_original.csv')
df_his = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8')
df_his

,região,indicador,ano,qtd_unidades
0,Aricanduva-Formosa-Carrão,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
1,Itaim Paulista,11.01.03 Número de Unidades Habitacionais entr...,2021,600.0
2,Perus,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
3,Pinheiros,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
4,Itaquera,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
...,...,...,...,...
91,M'Boi Mirim,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
92,Mooca,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
93,Jaçanã-Tremembé,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
94,Lapa,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0


In [11]:
filename = path.join(input_dir, 'tpu_emitido_original.csv')
df_tpu = pd.read_csv(filename,
            sep=';',
            decimal=',',
            encoding='utf8')
df_tpu

,indicador,ano,qtd_termos
0,05.0a.04 Número de termos de Permissão de Uso ...,2021,285.0
1,05.0a.04 Número de termos de Permissão de Uso ...,2022,293.0
2,05.0a.04 Número de termos de Permissão de Uso ...,2023,384.0


In [12]:
filename = path.join(input_dir, 'favelas_original.gpkg')
df_favelas = gpd.read_file(filename)
df_favelas

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente ocupado - com entrevista,Particular permanente ocupado - sem entrevista,Particular permanente não ocupado,Particular permanente não ocupado - vago,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,411,411,411,...,213,179,19,18,1,0,0,0,0,"MULTIPOLYGON (((359999.294 7400461.018, 359969..."
1,35503082222,Avenida Arraias do Araguaia,35,São Paulo,SP,3550308,São Paulo,133,133,31,...,17,1,13,11,2,102,0,0,0,"MULTIPOLYGON (((346540.261 7391411.884, 346562..."
2,35503081646,Pavanas,35,São Paulo,SP,3550308,São Paulo,58,58,58,...,50,5,3,3,0,0,0,0,0,"MULTIPOLYGON (((331961.694 7381437.08, 331927...."
3,35503081067,Jardim São Carlos / Gleba 2,35,São Paulo,SP,3550308,São Paulo,496,496,496,...,312,153,31,29,2,0,0,0,0,"MULTIPOLYGON (((356033.213 7393960.866, 356024..."
4,35503081121,Jardim Jaraguá II,35,São Paulo,SP,3550308,São Paulo,118,118,118,...,71,23,24,19,5,0,0,0,0,"MULTIPOLYGON (((317018.47 7405175.928, 316992...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1354,35503080507,São Francisco,35,São Paulo,SP,3550308,São Paulo,413,413,413,...,350,32,31,28,3,0,0,0,0,"MULTIPOLYGON (((332139.677 7377746.652, 332133..."
1355,35503080372,Jardim das Rosas,35,São Paulo,SP,3550308,São Paulo,445,444,444,...,357,48,39,37,2,0,1,0,1,"MULTIPOLYGON (((317588.774 7382183.16, 317570...."
1356,35503080334,Souza Dantas,35,São Paulo,SP,3550308,São Paulo,742,742,734,...,576,36,122,114,8,8,0,0,0,"MULTIPOLYGON (((330614.329 7383829.698, 330595..."
1357,35503081241,Sete de Setembro II,35,São Paulo,SP,3550308,São Paulo,461,461,461,...,398,33,30,26,4,0,0,0,0,"MULTIPOLYGON (((328106.645 7369890.12, 328127...."


Como vamos precisar fazer a interseção espacial entre os dados de favelas e o mapa de subprefeituras, vamos carregar também o mapa de subprefeituras, que foi salvo pelo notebook `01 areas de risco - extração.ipynb`.

In [13]:
filename = path.join(input_dir, 'subprefeituras_original.gpkg')
gdf_subs = gpd.read_file(filename)
gdf_subs

,id,cd_identificador_subprefeitura,cd_subprefeitura,nm_subprefeitura,tx_escala,sg_fonte_original,dt_criacao,cd_tipo_discrepancia,dt_atualizacao,cd_usuario_atualizacao,sg_subprefeitura,qt_area_quilometro,qt_area_metro,geometry
0,subprefeitura.1,1,02,PIRITUBA-JARAGUA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.042000+00:00,,PJ,55,5.502102e+07,"POLYGON ((318663.925 7404127.712, 318663.251 7..."
1,subprefeitura.2,2,03,FREGUESIA-BRASILANDIA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.050000+00:00,,FO,32,3.198020e+07,"POLYGON ((327340.628 7399133.313, 327331.514 7..."
2,subprefeitura.3,3,04,CASA VERDE-CACHOEIRINHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.956000+00:00,,CV,27,2.723234e+07,"POLYGON ((329084.795 7402363.669, 329086.123 7..."
3,subprefeitura.4,4,05,SANTANA-TUCURUVI,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.068000+00:00,,ST,36,3.578252e+07,"POLYGON ((334076.366 7398045.594, 334074.986 7..."
4,subprefeitura.5,5,06,JACANA-TREMEMBE,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.950000+00:00,,JT,65,6.511566e+07,"POLYGON ((335167.648 7404409.048, 335167.247 7..."
5,subprefeitura.6,6,07,VILA MARIA-VILA GUILHERME,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.073000+00:00,,MG,27,2.689922e+07,"POLYGON ((336762.078 7401144.267, 336794.867 7..."
6,subprefeitura.7,7,21,PENHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.055000+00:00,,PE,40,4.042924e+07,"POLYGON ((346801.065 7402842.892, 346800.103 7..."
7,subprefeitura.8,8,22,ERMELINO MATARAZZO,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.081000+00:00,,EM,16,1.596639e+07,"POLYGON ((349116.316 7399473.221, 349133.905 7..."
8,subprefeitura.9,9,23,SAO MIGUEL,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.944000+00:00,,MP,26,2.615923e+07,"POLYGON ((352480.323 7397515.871, 352477.052 7..."
9,subprefeitura.10,10,24,ITAIM PAULISTA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.967000+00:00,,IT,22,2.160382e+07,"POLYGON ((356423.064 7397223.44, 356418.471 73..."


# Transformação e padronização

Nos indicadores de habitação, apenas o ano e as subprefeituras são presentes em vários arquivos. Mas, além disso, os arquivos precisarão de tratamentos específicos. Vamos começar com os mais simples, onde é necessário apenas padronizar as subprefeituras.

Primeiro, vamos carregar os dados de subprefeituras que serão utilizados no Qlik Sense.

## CSV de Subprefeituras do Qlik

In [14]:
url_subs = environ.get('CSV_SUBPREFEITURAS_QLIK')
df_subs = pd.read_csv(url_subs)
df_subs

,sub.CODIGO,sub.NOME,sub.POLIGONO,total_de_pessoas
0,1,PERUS,"[[[[-46.7082012617476,-23.4165818252815],[-46....",163076
1,2,PIRITUBA-JARAGUA,"[[[[-46.7753669019781,-23.4628032705496],[-46....",480225
2,3,FREGUESIA-BRASILANDIA,"[[[[-46.6910223492151,-23.5088410763715],[-46....",380513
3,4,CASA VERDE-CACHOEIRINHA,"[[[[-46.673577305455,-23.479858692024],[-46.67...",306275
4,5,SANTANA-TUCURUVI,"[[[[-46.6251942372966,-23.5193645522374],[-46....",318913
5,6,JACANA-TREMEMBE,"[[[[-46.6138088697146,-23.4620183373337],[-46....",283892
6,7,VILA MARIA-VILA GUILHERME,"[[[[-46.5985584306269,-23.4916579527685],[-46....",276069
7,8,LAPA,"[[[[-46.7475634318888,-23.5587659797062],[-46....",338347
8,9,SE,"[[[[-46.6634017492882,-23.5366395157441],[-46....",423536
9,10,BUTANTA,"[[[[-46.7579053376809,-23.5510466758162],[-46....",468522


In [15]:
df_subs = df_subs[['sub.CODIGO', 'sub.NOME']]
df_subs

,sub.CODIGO,sub.NOME
0,1,PERUS
1,2,PIRITUBA-JARAGUA
2,3,FREGUESIA-BRASILANDIA
3,4,CASA VERDE-CACHOEIRINHA
4,5,SANTANA-TUCURUVI
5,6,JACANA-TREMEMBE
6,7,VILA MARIA-VILA GUILHERME
7,8,LAPA
8,9,SE
9,10,BUTANTA


## Chave composta subprefeitura-ano

Como 3 tabelas possuem valores para mais de um ano, também vale a pena a criação de uma chave composta entre subprefeitura e ano. A tabela que possui mais períodos é a tabela da meta 12 do Programa de Metas, com os anos de 2021, 2022, 2023 e 2024. Vamos criar uma tabela com o produto cartesiano entre subprefeituras e anos.

In [16]:
df_subs_ano = (
    df_subs[['sub.NOME']]
    .merge(pd.Series(data=[2021, 2022, 2023, 2024], name='ano'),
           how='cross')
)

df_subs_ano.loc[:, 'subprefeitura-ano'] = (
    df_subs_ano.loc[:, 'sub.NOME'] + ' | ' + df_subs_ano.loc[:, 'ano'].astype(str)
)

df_subs_ano

,sub.NOME,ano,subprefeitura-ano
0,PERUS,2021,PERUS | 2021
1,PERUS,2022,PERUS | 2022
2,PERUS,2023,PERUS | 2023
3,PERUS,2024,PERUS | 2024
4,PIRITUBA-JARAGUA,2021,PIRITUBA-JARAGUA | 2021
...,...,...,...
123,CIDADE TIRADENTES,2024,CIDADE TIRADENTES | 2024
124,SAPOPEMBA,2021,SAPOPEMBA | 2021
125,SAPOPEMBA,2022,SAPOPEMBA | 2022
126,SAPOPEMBA,2023,SAPOPEMBA | 2023


## PdM - Meta 12: Prover 49.000 moradias de interesse social

In [17]:
df_meta_12

,Número da Meta,Descrição,Área,Subprefeitura,2021,2022,2023,2024
0,12,Prover 49.000 moradias de interesse social.,Habitação,Butantã,216.0,546.0,3081.0,7261.0
1,12,Prover 49.000 moradias de interesse social.,Habitação,Campo Limpo,1316.0,1748.0,1748.0,2161.0
2,12,Prover 49.000 moradias de interesse social.,Habitação,Capela do Socorro,562.0,2040.0,2124.0,2140.0
3,12,Prover 49.000 moradias de interesse social.,Habitação,Casa Verde,0.0,108.0,128.0,128.0
4,12,Prover 49.000 moradias de interesse social.,Habitação,Cidade Ademar,0.0,8.0,78.0,78.0
5,12,Prover 49.000 moradias de interesse social.,Habitação,Cidade Tiradentes,50.0,50.0,347.0,507.0
6,12,Prover 49.000 moradias de interesse social.,Habitação,Ermelino Matarazzo,0.0,0.0,0.0,266.0
7,12,Prover 49.000 moradias de interesse social.,Habitação,Freguesia/Brasilândia,228.0,228.0,228.0,502.0
8,12,Prover 49.000 moradias de interesse social.,Habitação,Guaianases,0.0,968.0,1122.0,1322.0
9,12,Prover 49.000 moradias de interesse social.,Habitação,Ipiranga,1694.0,1965.0,2205.0,2685.0


In [18]:
df_meta_12 = df_meta_12.loc[:, ['Subprefeitura', '2021', '2022', '2023', '2024']]

df_meta_12

,Subprefeitura,2021,2022,2023,2024
0,Butantã,216.0,546.0,3081.0,7261.0
1,Campo Limpo,1316.0,1748.0,1748.0,2161.0
2,Capela do Socorro,562.0,2040.0,2124.0,2140.0
3,Casa Verde,0.0,108.0,128.0,128.0
4,Cidade Ademar,0.0,8.0,78.0,78.0
5,Cidade Tiradentes,50.0,50.0,347.0,507.0
6,Ermelino Matarazzo,0.0,0.0,0.0,266.0
7,Freguesia/Brasilândia,228.0,228.0,228.0,502.0
8,Guaianases,0.0,968.0,1122.0,1322.0
9,Ipiranga,1694.0,1965.0,2205.0,2685.0


In [19]:
df_meta_12 = df_meta_12.melt('Subprefeitura',
                var_name='ano',
                value_name='qtd_unidades_acumulado')

df_meta_12

,Subprefeitura,ano,qtd_unidades_acumulado
0,Butantã,2021,216.0
1,Campo Limpo,2021,1316.0
2,Capela do Socorro,2021,562.0
3,Casa Verde,2021,0.0
4,Cidade Ademar,2021,0.0
...,...,...,...
111,São Mateus,2024,3504.0
112,Sapopemba,2024,216.0
113,Sé,2024,14744.0
114,Vila Maria/Vila Guilherme,2024,2020.0


Como os valores foram divulgados no acumulado entre 2021 e 2024, vamos calcular o incremento de cada ano antes de carregar os dados no Qlik, mas mantendo as duas colunas.

In [20]:
df_meta_12['qtd_unidades'] = df_meta_12.groupby('Subprefeitura')['qtd_unidades_acumulado'].diff()
df_meta_12

,Subprefeitura,ano,qtd_unidades_acumulado,qtd_unidades
0,Butantã,2021,216.0,NaN
1,Campo Limpo,2021,1316.0,NaN
2,Capela do Socorro,2021,562.0,NaN
3,Casa Verde,2021,0.0,NaN
4,Cidade Ademar,2021,0.0,NaN
...,...,...,...,...
111,São Mateus,2024,3504.0,472.0
112,Sapopemba,2024,216.0,0.0
113,Sé,2024,14744.0,190.0
114,Vila Maria/Vila Guilherme,2024,2020.0,768.0


In [21]:
df_meta_12.loc[df_meta_12['ano']=='2021', 'qtd_unidades'] = (
    df_meta_12.loc[df_meta_12['ano']=='2021', 'qtd_unidades_acumulado'])
df_meta_12

,Subprefeitura,ano,qtd_unidades_acumulado,qtd_unidades
0,Butantã,2021,216.0,216.0
1,Campo Limpo,2021,1316.0,1316.0
2,Capela do Socorro,2021,562.0,562.0
3,Casa Verde,2021,0.0,0.0
4,Cidade Ademar,2021,0.0,0.0
...,...,...,...,...
111,São Mateus,2024,3504.0,472.0
112,Sapopemba,2024,216.0,0.0
113,Sé,2024,14744.0,190.0
114,Vila Maria/Vila Guilherme,2024,2020.0,768.0


Finalmente, vamos criar uma coluna com os nomes padronizados de subprefeituras.

In [22]:
subs_meta_12 = df_meta_12['Subprefeitura'].apply(unidecode).unique().tolist()
subs_meta_12.sort()
subs_meta_12

['Butanta',
 'Campo Limpo',
 'Capela do Socorro',
 'Casa Verde',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia/Brasilandia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jacana/Tremembe',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus/Anhanguera',
 'Pinheiros',
 'Pirituba/Jaragua',
 'Santana/Tucuruvi',
 'Santo Amaro',
 'Sao Mateus',
 'Sapopemba',
 'Se',
 'Vila Maria/Vila Guilherme',
 'Vila Prudente']

In [23]:
subs_qlik = df_subs['sub.NOME'].unique().tolist()
subs_qlik.sort()
subs_qlik

['ARICANDUVA-FORMOSA-CARRAO',
 'BUTANTA',
 'CAMPO LIMPO',
 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR',
 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA',
 'GUAIANASES',
 'IPIRANGA',
 'ITAIM PAULISTA',
 'ITAQUERA',
 'JABAQUARA',
 'JACANA-TREMEMBE',
 'LAPA',
 'M BOI MIRIM',
 'MOOCA',
 'PARELHEIROS',
 'PENHA',
 'PERUS',
 'PINHEIROS',
 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI',
 'SANTO AMARO',
 'SAO MATEUS',
 'SAO MIGUEL',
 'SAPOPEMBA',
 'SE',
 'VILA MARIA-VILA GUILHERME',
 'VILA MARIANA',
 'VILA PRUDENTE']

In [24]:
len(subs_meta_12)

29

Vemos que existem 3 subprefeituras faltantes no dataframe da meta 12. Vamos avaliar quais podem ser. Numa inspeção detalhada vemos que faltam `ARICANDUVA-FORMOSA-CARRAO`, `SAO MIGUEL` e `VILA MARIANA`. Vamos criar uma cópia da lista de subs do qlik adaptada à meta 12.

In [25]:
subs_qlik_meta_12 = subs_qlik.copy()
subs_qlik_meta_12.remove('ARICANDUVA-FORMOSA-CARRAO')
subs_qlik_meta_12.remove('SAO MIGUEL')
subs_qlik_meta_12.remove('VILA MARIANA')
subs_qlik_meta_12

['BUTANTA',
 'CAMPO LIMPO',
 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR',
 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA',
 'GUAIANASES',
 'IPIRANGA',
 'ITAIM PAULISTA',
 'ITAQUERA',
 'JABAQUARA',
 'JACANA-TREMEMBE',
 'LAPA',
 'M BOI MIRIM',
 'MOOCA',
 'PARELHEIROS',
 'PENHA',
 'PERUS',
 'PINHEIROS',
 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI',
 'SANTO AMARO',
 'SAO MATEUS',
 'SAPOPEMBA',
 'SE',
 'VILA MARIA-VILA GUILHERME',
 'VILA PRUDENTE']

In [26]:
mapper_meta_12 = {
    o: q
    for o, q in zip(subs_meta_12, subs_qlik_meta_12)
}

mapper_meta_12

{'Butanta': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia/Brasilandia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jacana/Tremembe': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus/Anhanguera': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba/Jaragua': 'PIRITUBA-JARAGUA',
 'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'Sao Mateus': 'SAO MATEUS',
 'Sapopemba': 'SAPOPEMBA',
 'Se': 'SE',
 'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
 'Vila Prudente': 'VILA PRUDENTE'}

In [27]:
df_meta_12.insert(1,
                  'sub.NOME',
                  df_meta_12['Subprefeitura'].apply(unidecode).map(mapper_meta_12))
df_meta_12

,Subprefeitura,sub.NOME,ano,qtd_unidades_acumulado,qtd_unidades
0,Butantã,BUTANTA,2021,216.0,216.0
1,Campo Limpo,CAMPO LIMPO,2021,1316.0,1316.0
2,Capela do Socorro,CAPELA DO SOCORRO,2021,562.0,562.0
3,Casa Verde,CASA VERDE-CACHOEIRINHA,2021,0.0,0.0
4,Cidade Ademar,CIDADE ADEMAR,2021,0.0,0.0
...,...,...,...,...,...
111,São Mateus,SAO MATEUS,2024,3504.0,472.0
112,Sapopemba,SAPOPEMBA,2024,216.0,0.0
113,Sé,SE,2024,14744.0,190.0
114,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,2024,2020.0,768.0


In [28]:
df_meta_12['qtd_unidades'] = df_meta_12['qtd_unidades'].astype(int)
df_meta_12['qtd_unidades_acumulado'] = df_meta_12['qtd_unidades_acumulado'].astype(int)
df_meta_12['ano'] = df_meta_12['ano'].astype(int)
df_meta_12

,Subprefeitura,sub.NOME,ano,qtd_unidades_acumulado,qtd_unidades
0,Butantã,BUTANTA,2021,216,216
1,Campo Limpo,CAMPO LIMPO,2021,1316,1316
2,Capela do Socorro,CAPELA DO SOCORRO,2021,562,562
3,Casa Verde,CASA VERDE-CACHOEIRINHA,2021,0,0
4,Cidade Ademar,CIDADE ADEMAR,2021,0,0
...,...,...,...,...,...
111,São Mateus,SAO MATEUS,2024,3504,472
112,Sapopemba,SAPOPEMBA,2024,216,0
113,Sé,SE,2024,14744,190
114,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,2024,2020,768


In [29]:
df_meta_12 = df_meta_12.merge(df_subs_ano,
                              how='left',
                              on=['sub.NOME', 'ano'])

df_meta_12

,Subprefeitura,sub.NOME,ano,qtd_unidades_acumulado,qtd_unidades,subprefeitura-ano
0,Butantã,BUTANTA,2021,216,216,BUTANTA | 2021
1,Campo Limpo,CAMPO LIMPO,2021,1316,1316,CAMPO LIMPO | 2021
2,Capela do Socorro,CAPELA DO SOCORRO,2021,562,562,CAPELA DO SOCORRO | 2021
3,Casa Verde,CASA VERDE-CACHOEIRINHA,2021,0,0,CASA VERDE-CACHOEIRINHA | 2021
4,Cidade Ademar,CIDADE ADEMAR,2021,0,0,CIDADE ADEMAR | 2021
...,...,...,...,...,...,...
111,São Mateus,SAO MATEUS,2024,3504,472,SAO MATEUS | 2024
112,Sapopemba,SAPOPEMBA,2024,216,0,SAPOPEMBA | 2024
113,Sé,SE,2024,14744,190,SE | 2024
114,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,2024,2020,768,VILA MARIA-VILA GUILHERME | 2024


## Produção de habitação de interesse social

In [30]:
df_his

,região,indicador,ano,qtd_unidades
0,Aricanduva-Formosa-Carrão,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
1,Itaim Paulista,11.01.03 Número de Unidades Habitacionais entr...,2021,600.0
2,Perus,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
3,Pinheiros,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
4,Itaquera,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
...,...,...,...,...
91,M'Boi Mirim,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
92,Mooca,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
93,Jaçanã-Tremembé,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
94,Lapa,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0


In [31]:
subs_his = df_his['região'].apply(unidecode).unique().tolist()
subs_his.sort()
subs_his

['Aricanduva-Formosa-Carrao',
 'Butanta',
 'Campo Limpo',
 'Capela Do Socorro',
 'Casa Verde-Cachoeirinha',
 'Cidade Ademar',
 'Cidade Tiradentes',
 'Ermelino Matarazzo',
 'Freguesia-Brasilandia',
 'Guaianases',
 'Ipiranga',
 'Itaim Paulista',
 'Itaquera',
 'Jabaquara',
 'Jacana-Tremembe',
 'Lapa',
 "M'Boi Mirim",
 'Mooca',
 'Parelheiros',
 'Penha',
 'Perus',
 'Pinheiros',
 'Pirituba-Jaragua',
 'Santana-Tucuruvi',
 'Santo Amaro',
 'Sao Mateus',
 'Sao Miguel',
 'Sapopemba',
 'Se',
 'Vila Maria-Vila Guilherme',
 'Vila Mariana',
 'Vila Prudente']

In [32]:
mapper_his = {
    s: q for s, q in zip(subs_his, subs_qlik)
}

mapper_his

{'Aricanduva-Formosa-Carrao': 'ARICANDUVA-FORMOSA-CARRAO',
 'Butanta': 'BUTANTA',
 'Campo Limpo': 'CAMPO LIMPO',
 'Capela Do Socorro': 'CAPELA DO SOCORRO',
 'Casa Verde-Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Cidade Ademar': 'CIDADE ADEMAR',
 'Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
 'Freguesia-Brasilandia': 'FREGUESIA-BRASILANDIA',
 'Guaianases': 'GUAIANASES',
 'Ipiranga': 'IPIRANGA',
 'Itaim Paulista': 'ITAIM PAULISTA',
 'Itaquera': 'ITAQUERA',
 'Jabaquara': 'JABAQUARA',
 'Jacana-Tremembe': 'JACANA-TREMEMBE',
 'Lapa': 'LAPA',
 "M'Boi Mirim": 'M BOI MIRIM',
 'Mooca': 'MOOCA',
 'Parelheiros': 'PARELHEIROS',
 'Penha': 'PENHA',
 'Perus': 'PERUS',
 'Pinheiros': 'PINHEIROS',
 'Pirituba-Jaragua': 'PIRITUBA-JARAGUA',
 'Santana-Tucuruvi': 'SANTANA-TUCURUVI',
 'Santo Amaro': 'SANTO AMARO',
 'Sao Mateus': 'SAO MATEUS',
 'Sao Miguel': 'SAO MIGUEL',
 'Sapopemba': 'SAPOPEMBA',
 'Se': 'SE',
 'Vila Maria-Vila Guilherme': 'VILA MARIA-VILA GUILHERME',


In [33]:
df_his.insert(1,
                  'sub.NOME',
                  df_his['região'].apply(unidecode).map(mapper_his))
df_his

,região,sub.NOME,indicador,ano,qtd_unidades
0,Aricanduva-Formosa-Carrão,ARICANDUVA-FORMOSA-CARRAO,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
1,Itaim Paulista,ITAIM PAULISTA,11.01.03 Número de Unidades Habitacionais entr...,2021,600.0
2,Perus,PERUS,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
3,Pinheiros,PINHEIROS,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
4,Itaquera,ITAQUERA,11.01.03 Número de Unidades Habitacionais entr...,2021,0.0
...,...,...,...,...,...
91,M'Boi Mirim,M BOI MIRIM,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
92,Mooca,MOOCA,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
93,Jaçanã-Tremembé,JACANA-TREMEMBE,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0
94,Lapa,LAPA,11.01.03 Número de Unidades Habitacionais entr...,2023,0.0


In [34]:
df_his['qtd_unidades'] = df_his['qtd_unidades'].astype(int)
df_his['ano'] = df_his['ano'].astype(int)
df_his

,região,sub.NOME,indicador,ano,qtd_unidades
0,Aricanduva-Formosa-Carrão,ARICANDUVA-FORMOSA-CARRAO,11.01.03 Número de Unidades Habitacionais entr...,2021,0
1,Itaim Paulista,ITAIM PAULISTA,11.01.03 Número de Unidades Habitacionais entr...,2021,600
2,Perus,PERUS,11.01.03 Número de Unidades Habitacionais entr...,2021,0
3,Pinheiros,PINHEIROS,11.01.03 Número de Unidades Habitacionais entr...,2021,0
4,Itaquera,ITAQUERA,11.01.03 Número de Unidades Habitacionais entr...,2021,0
...,...,...,...,...,...
91,M'Boi Mirim,M BOI MIRIM,11.01.03 Número de Unidades Habitacionais entr...,2023,0
92,Mooca,MOOCA,11.01.03 Número de Unidades Habitacionais entr...,2023,0
93,Jaçanã-Tremembé,JACANA-TREMEMBE,11.01.03 Número de Unidades Habitacionais entr...,2023,0
94,Lapa,LAPA,11.01.03 Número de Unidades Habitacionais entr...,2023,0


In [35]:
df_his = df_his.merge(df_subs_ano,
                              how='left',
                              on=['sub.NOME', 'ano'])

df_his

,região,sub.NOME,indicador,ano,qtd_unidades,subprefeitura-ano
0,Aricanduva-Formosa-Carrão,ARICANDUVA-FORMOSA-CARRAO,11.01.03 Número de Unidades Habitacionais entr...,2021,0,ARICANDUVA-FORMOSA-CARRAO | 2021
1,Itaim Paulista,ITAIM PAULISTA,11.01.03 Número de Unidades Habitacionais entr...,2021,600,ITAIM PAULISTA | 2021
2,Perus,PERUS,11.01.03 Número de Unidades Habitacionais entr...,2021,0,PERUS | 2021
3,Pinheiros,PINHEIROS,11.01.03 Número de Unidades Habitacionais entr...,2021,0,PINHEIROS | 2021
4,Itaquera,ITAQUERA,11.01.03 Número de Unidades Habitacionais entr...,2021,0,ITAQUERA | 2021
...,...,...,...,...,...,...
91,M'Boi Mirim,M BOI MIRIM,11.01.03 Número de Unidades Habitacionais entr...,2023,0,M BOI MIRIM | 2023
92,Mooca,MOOCA,11.01.03 Número de Unidades Habitacionais entr...,2023,0,MOOCA | 2023
93,Jaçanã-Tremembé,JACANA-TREMEMBE,11.01.03 Número de Unidades Habitacionais entr...,2023,0,JACANA-TREMEMBE | 2023
94,Lapa,LAPA,11.01.03 Número de Unidades Habitacionais entr...,2023,0,LAPA | 2023


## Número de termos de Permissão de Uso (TPU) emitidos em nome da mulher da familia

In [36]:
df_tpu['qtd_termos'] = df_tpu['qtd_termos'].astype(int)
df_tpu['ano'] = df_tpu['ano'].astype(int)
df_tpu

,indicador,ano,qtd_termos
0,05.0a.04 Número de termos de Permissão de Uso ...,2021,285
1,05.0a.04 Número de termos de Permissão de Uso ...,2022,293
2,05.0a.04 Número de termos de Permissão de Uso ...,2023,384


## Número de domicílios em favelas

Como os domicílios em favelas não possuem uma coluna identificando a subprefeitura, precisamos fazer a interseção espacial entre os dados de favelas e o mapa de subprefeituras.

Primeiro, vamos chegar se existem favelas que estão dentro de mais de uma subprefeitura.

In [37]:
df_fav_sub = df_favelas.overlay(gdf_subs, how='intersection')
df_fav_sub.iloc[:,0].describe()

count            1457
unique           1359
top       35503081550
freq                3
Name: cd_fcu, dtype: object

Existem 8 registros duplicados, o que indica que algumas favelas estão em mais de uma subprefeitura. Portanto, um spatial join simples não funcionará. Vamos fazer a interseção espacial entre os dois dataframes.

### Interpolação ponderada por área

In [38]:
df_fav_sub = areal_weighted_interpolation(
    left=df_favelas,
    right=gdf_subs,
    right_id_col='nm_subprefeitura',
    original_var_name='Particular permanente',
    final_var_name='domicilios_particulares_permanentes_favela'
)
df_fav_sub

,id,cd_identificador_subprefeitura,cd_subprefeitura,nm_subprefeitura,tx_escala,sg_fonte_original,dt_criacao,cd_tipo_discrepancia,dt_atualizacao,cd_usuario_atualizacao,sg_subprefeitura,qt_area_quilometro,qt_area_metro,geometry,domicilios_particulares_permanentes_favela
0,subprefeitura.1,1,02,PIRITUBA-JARAGUA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.042000+00:00,,PJ,55,5.502102e+07,"POLYGON ((318663.925 7404127.712, 318663.251 7...",26713
1,subprefeitura.2,2,03,FREGUESIA-BRASILANDIA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.050000+00:00,,FO,32,3.198020e+07,"POLYGON ((327340.628 7399133.313, 327331.514 7...",32156
2,subprefeitura.3,3,04,CASA VERDE-CACHOEIRINHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.956000+00:00,,CV,27,2.723234e+07,"POLYGON ((329084.795 7402363.669, 329086.123 7...",11697
3,subprefeitura.4,4,05,SANTANA-TUCURUVI,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.068000+00:00,,ST,36,3.578252e+07,"POLYGON ((334076.366 7398045.594, 334074.986 7...",759
4,subprefeitura.5,5,06,JACANA-TREMEMBE,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.950000+00:00,,JT,65,6.511566e+07,"POLYGON ((335167.648 7404409.048, 335167.247 7...",31502
5,subprefeitura.6,6,07,VILA MARIA-VILA GUILHERME,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.073000+00:00,,MG,27,2.689922e+07,"POLYGON ((336762.078 7401144.267, 336794.867 7...",9078
6,subprefeitura.7,7,21,PENHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.055000+00:00,,PE,40,4.042924e+07,"POLYGON ((346801.065 7402842.892, 346800.103 7...",14156
7,subprefeitura.8,8,22,ERMELINO MATARAZZO,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.081000+00:00,,EM,16,1.596639e+07,"POLYGON ((349116.316 7399473.221, 349133.905 7...",9023
8,subprefeitura.9,9,23,SAO MIGUEL,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.944000+00:00,,MP,26,2.615923e+07,"POLYGON ((352480.323 7397515.871, 352477.052 7...",26972
9,subprefeitura.10,10,24,ITAIM PAULISTA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.967000+00:00,,IT,22,2.160382e+07,"POLYGON ((356423.064 7397223.44, 356418.471 73...",11250


In [39]:
df_favelas['Particular permanente'].sum()

np.int64(649765)

In [40]:
df_fav_sub['domicilios_particulares_permanentes_favela'].sum() 

np.int64(649106)

In [41]:
df_favelas['Particular permanente'].sum()-df_fav_sub['domicilios_particulares_permanentes_favela'].sum()

np.int64(659)

In [42]:
1-df_fav_sub['domicilios_particulares_permanentes_favela'].sum()/df_favelas['Particular permanente'].sum()

np.float64(0.0010142128307926157)

Cerca de 659 domicílios em favelas não foram alocados a nenhuma subprefeitura. Isso representa apenas 0,1% do total de domicílios em favelas, então não é uma perda grave. Porém, o caso mais provável é de que parte dessas favelas estejam fora do limite das subprefeituras, o que não deveria ocorrer com os dados de favelas filtrados apenas para o município de São Paulo. Vamos investigar.

### Avaliando domicílios não alocados

In [43]:
df_favelas_fora = df_favelas.overlay(gdf_subs, how='difference', keep_geom_type=True)
df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente ocupado - com entrevista,Particular permanente ocupado - sem entrevista,Particular permanente não ocupado,Particular permanente não ocupado - vago,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,36,30,11,11,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7..."
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,49,16,8,8,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7..."
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,838,143,54,51,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060..."
3,35503080577,Jardim Campo de Fora / Santo Antônio,35,São Paulo,SP,3550308,São Paulo,1118,1118,1118,...,970,51,97,80,17,0,0,0,0,"MULTIPOLYGON (((320651.062 7382080.936, 320653..."
4,35503082102,Comunidade Córrego do Oratório,35,São Paulo,SP,3550308,São Paulo,129,129,126,...,115,1,10,10,0,3,0,0,0,"MULTIPOLYGON (((343577.956 7387642.187, 343568..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,35503082181,Jardim São Raimundo / Dom Angélico,35,São Paulo,SP,3550308,São Paulo,857,857,857,...,714,24,119,104,15,0,0,0,0,"POLYGON ((358918.798 7391538.999, 358940.94 73..."
108,35503080560,Vila Constância,35,São Paulo,SP,3550308,São Paulo,141,141,141,...,101,28,12,11,1,0,0,0,0,"MULTIPOLYGON (((346855.758 7399497.616, 346859..."
109,35503080832,Chazinho,35,São Paulo,SP,3550308,São Paulo,301,301,300,...,247,28,25,22,3,1,0,0,0,"POLYGON ((353276.461 7397562.926, 353373.892 7..."
110,35503081550,Rufino Fernandes Ivivari,35,São Paulo,SP,3550308,São Paulo,250,250,249,...,195,32,22,21,1,1,0,0,0,"MULTIPOLYGON (((346245.386 7390356.726, 346247..."


Como era esperado, existem favelas que estão parcialmente fora do limite do município de São Paulo. Vamos inspecionar visualmente onde estão essas favelas.

In [44]:
m = gdf_subs.explore(
    tiles='CartoDB positron',
    tooltip=True,
    popup=True,
    style_kwds=dict(color='grey', fill=False))

m = df_favelas_fora.explore(
    m=m,
    legend=True,
    tooltip=True,
    popup=True)

m

De fato, existem favelas que estão parcialmente fora do limite do município de São Paulo, e são onde existem as áreas mais significativas de favelas fora das subprefeituras. Porém, também existem casos áreas de favelas que estão completamente dentro do município de São Paulo, mas que não foram alocadas a nenhuma subprefeitura por se situarem exatamente na divisa entre subprefeituras, mas provavelmente isso não representa uma área grande o suficiente para distorcer as estimativas.

### Identificando áreas fora dos limites das subprefeituras

De todo modo, vamos calcular a proporção de domicílios dessas áreas que não foram alocados a nenhuma subprefeitura, e aplicar essa proporção para redistribuir os domicílios não alocados entre as subprefeituras. Com a proporção de domicílios calculada, podemos atribuir os domicílios não alocados à subprefeitura mais próxima de cada área de favela não alocada.

In [45]:
df_favelas_fora['area_total'] = df_favelas_fora.apply(
    lambda row: df_favelas.query(f'cd_fcu=="{row.cd_fcu}"').geometry.area.iloc[0],
    axis=1)

df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente ocupado - sem entrevista,Particular permanente não ocupado,Particular permanente não ocupado - vago,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,30,11,11,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,16,8,8,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,143,54,51,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468
3,35503080577,Jardim Campo de Fora / Santo Antônio,35,São Paulo,SP,3550308,São Paulo,1118,1118,1118,...,51,97,80,17,0,0,0,0,"MULTIPOLYGON (((320651.062 7382080.936, 320653...",85242.615788
4,35503082102,Comunidade Córrego do Oratório,35,São Paulo,SP,3550308,São Paulo,129,129,126,...,1,10,10,0,3,0,0,0,"MULTIPOLYGON (((343577.956 7387642.187, 343568...",24891.096133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,35503082181,Jardim São Raimundo / Dom Angélico,35,São Paulo,SP,3550308,São Paulo,857,857,857,...,24,119,104,15,0,0,0,0,"POLYGON ((358918.798 7391538.999, 358940.94 73...",117009.484528
108,35503080560,Vila Constância,35,São Paulo,SP,3550308,São Paulo,141,141,141,...,28,12,11,1,0,0,0,0,"MULTIPOLYGON (((346855.758 7399497.616, 346859...",10096.746864
109,35503080832,Chazinho,35,São Paulo,SP,3550308,São Paulo,301,301,300,...,28,25,22,3,1,0,0,0,"POLYGON ((353276.461 7397562.926, 353373.892 7...",38925.656078
110,35503081550,Rufino Fernandes Ivivari,35,São Paulo,SP,3550308,São Paulo,250,250,249,...,32,22,21,1,1,0,0,0,"MULTIPOLYGON (((346245.386 7390356.726, 346247...",27123.844942


In [46]:
df_favelas_fora['area_fora'] = df_favelas_fora.geometry.area
df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente não ocupado,Particular permanente não ocupado - vago,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,11,11,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,9.905364e+02
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,8,8,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1.465398e+03
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,54,51,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4.221421e+03
3,35503080577,Jardim Campo de Fora / Santo Antônio,35,São Paulo,SP,3550308,São Paulo,1118,1118,1118,...,97,80,17,0,0,0,0,"MULTIPOLYGON (((320651.062 7382080.936, 320653...",85242.615788,2.256449e-08
4,35503082102,Comunidade Córrego do Oratório,35,São Paulo,SP,3550308,São Paulo,129,129,126,...,10,10,0,3,0,0,0,"MULTIPOLYGON (((343577.956 7387642.187, 343568...",24891.096133,6.425815e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,35503082181,Jardim São Raimundo / Dom Angélico,35,São Paulo,SP,3550308,São Paulo,857,857,857,...,119,104,15,0,0,0,0,"POLYGON ((358918.798 7391538.999, 358940.94 73...",117009.484528,3.547065e+03
108,35503080560,Vila Constância,35,São Paulo,SP,3550308,São Paulo,141,141,141,...,12,11,1,0,0,0,0,"MULTIPOLYGON (((346855.758 7399497.616, 346859...",10096.746864,6.727872e-10
109,35503080832,Chazinho,35,São Paulo,SP,3550308,São Paulo,301,301,300,...,25,22,3,1,0,0,0,"POLYGON ((353276.461 7397562.926, 353373.892 7...",38925.656078,1.200484e-09
110,35503081550,Rufino Fernandes Ivivari,35,São Paulo,SP,3550308,São Paulo,250,250,249,...,22,21,1,1,0,0,0,"MULTIPOLYGON (((346245.386 7390356.726, 346247...",27123.844942,3.363963e-04


In [47]:
df_favelas_fora['peso'] = (
    df_favelas_fora['area_fora'] / df_favelas_fora['area_total']
)
df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente não ocupado - vago,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora,peso
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,11,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,9.905364e+02,4.738530e-02
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,8,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1.465398e+03,1.294392e-01
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,51,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4.221421e+03,6.820447e-02
3,35503080577,Jardim Campo de Fora / Santo Antônio,35,São Paulo,SP,3550308,São Paulo,1118,1118,1118,...,80,17,0,0,0,0,"MULTIPOLYGON (((320651.062 7382080.936, 320653...",85242.615788,2.256449e-08,2.647091e-13
4,35503082102,Comunidade Córrego do Oratório,35,São Paulo,SP,3550308,São Paulo,129,129,126,...,10,0,3,0,0,0,"MULTIPOLYGON (((343577.956 7387642.187, 343568...",24891.096133,6.425815e+01,2.581572e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,35503082181,Jardim São Raimundo / Dom Angélico,35,São Paulo,SP,3550308,São Paulo,857,857,857,...,104,15,0,0,0,0,"POLYGON ((358918.798 7391538.999, 358940.94 73...",117009.484528,3.547065e+03,3.031434e-02
108,35503080560,Vila Constância,35,São Paulo,SP,3550308,São Paulo,141,141,141,...,11,1,0,0,0,0,"MULTIPOLYGON (((346855.758 7399497.616, 346859...",10096.746864,6.727872e-10,6.663405e-14
109,35503080832,Chazinho,35,São Paulo,SP,3550308,São Paulo,301,301,300,...,22,3,1,0,0,0,"POLYGON ((353276.461 7397562.926, 353373.892 7...",38925.656078,1.200484e-09,3.084044e-14
110,35503081550,Rufino Fernandes Ivivari,35,São Paulo,SP,3550308,São Paulo,250,250,249,...,21,1,1,0,0,0,"MULTIPOLYGON (((346245.386 7390356.726, 346247...",27123.844942,3.363963e-04,1.240224e-08


In [48]:
df_favelas_fora['domicilios_favela_fora'] = (
    df_favelas_fora['peso'] * df_favelas_fora['Particular permanente']
).round().astype(int)
df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora,peso,domicilios_favela_fora
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,9.905364e+02,4.738530e-02,4
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1.465398e+03,1.294392e-01,9
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4.221421e+03,6.820447e-02,71
3,35503080577,Jardim Campo de Fora / Santo Antônio,35,São Paulo,SP,3550308,São Paulo,1118,1118,1118,...,17,0,0,0,0,"MULTIPOLYGON (((320651.062 7382080.936, 320653...",85242.615788,2.256449e-08,2.647091e-13,0
4,35503082102,Comunidade Córrego do Oratório,35,São Paulo,SP,3550308,São Paulo,129,129,126,...,0,3,0,0,0,"MULTIPOLYGON (((343577.956 7387642.187, 343568...",24891.096133,6.425815e+01,2.581572e-03,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,35503082181,Jardim São Raimundo / Dom Angélico,35,São Paulo,SP,3550308,São Paulo,857,857,857,...,15,0,0,0,0,"POLYGON ((358918.798 7391538.999, 358940.94 73...",117009.484528,3.547065e+03,3.031434e-02,26
108,35503080560,Vila Constância,35,São Paulo,SP,3550308,São Paulo,141,141,141,...,1,0,0,0,0,"MULTIPOLYGON (((346855.758 7399497.616, 346859...",10096.746864,6.727872e-10,6.663405e-14,0
109,35503080832,Chazinho,35,São Paulo,SP,3550308,São Paulo,301,301,300,...,3,1,0,0,0,"POLYGON ((353276.461 7397562.926, 353373.892 7...",38925.656078,1.200484e-09,3.084044e-14,0
110,35503081550,Rufino Fernandes Ivivari,35,São Paulo,SP,3550308,São Paulo,250,250,249,...,1,1,0,0,0,"MULTIPOLYGON (((346245.386 7390356.726, 346247...",27123.844942,3.363963e-04,1.240224e-08,0


In [49]:
df_favelas_fora['domicilios_favela_fora'].sum()

np.int64(655)

De fato, 655 dos 659 domicílios não alocados estão em favelas que estão parcialmente fora do limite do município de São Paulo. Os outros 4 domicílios provavelmente são falhas de arredondamento. Antes de atribuir os domicílios não alocados, às subprefeituras mais próximas, vamos excluir do dataframe as áreas sem domicílios.

In [50]:
df_favelas_fora = df_favelas_fora[df_favelas_fora['domicilios_favela_fora']>0]
df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular permanente não ocupado - uso ocasional,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora,peso,domicilios_favela_fora
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,0,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,990.536419,0.047385,4
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,0,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1465.397866,0.129439,9
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,3,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4221.421108,0.068204,71
6,35503081196,São José,35,São Paulo,SP,3550308,São Paulo,143,143,143,...,0,0,0,0,0,"MULTIPOLYGON (((339568.383 7405014.852, 339569...",18526.246839,751.029470,0.040539,6
13,35503080925,Manuel Antônio Portella,35,São Paulo,SP,3550308,São Paulo,307,307,307,...,0,0,0,0,0,"POLYGON ((320297.363 7396761.169, 320304.738 7...",62199.018628,5149.310696,0.082788,25
14,35503081739,Fazenda da Juta III,35,São Paulo,SP,3550308,São Paulo,287,287,287,...,1,0,0,0,0,"MULTIPOLYGON (((347748.38 7386558.489, 347727....",27619.298504,821.096085,0.029729,9
19,35503080180,Liviero,35,São Paulo,SP,3550308,São Paulo,171,171,171,...,0,0,0,0,0,"POLYGON ((336911.453 7383279.915, 336906.821 7...",13069.809113,473.095572,0.036198,6
20,35503081859,Estrada de Poá,35,São Paulo,SP,3550308,São Paulo,42,42,42,...,2,0,0,0,0,"MULTIPOLYGON (((357877.212 7394414.473, 357874...",12915.034452,2454.411373,0.190043,8
21,35503080965,Francisco Capara,35,São Paulo,SP,3550308,São Paulo,165,165,165,...,0,0,0,0,0,"POLYGON ((358330.389 7395329.13, 358327.305 73...",17372.458232,5309.107608,0.305605,50
22,35503081959,Rua Professor Cosme Deodato Tadeu,35,São Paulo,SP,3550308,São Paulo,65,65,65,...,0,0,0,0,0,"POLYGON ((357612.105 7394799.908, 357615.945 7...",10215.616185,1424.359596,0.139430,9


### Atribuindo domicílios não alocados às subprefeituras mais próximas

In [51]:
df_favelas_fora = df_favelas_fora.sjoin_nearest(
    gdf_subs[['nm_subprefeitura', 'geometry']],
    how='left'
).drop(columns='index_right')

df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora,peso,domicilios_favela_fora,nm_subprefeitura
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,990.536419,0.047385,4,CIDADE TIRADENTES
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1465.397866,0.129439,9,GUAIANASES
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4221.421108,0.068204,71,BUTANTA
6,35503081196,São José,35,São Paulo,SP,3550308,São Paulo,143,143,143,...,0,0,0,0,"MULTIPOLYGON (((339568.383 7405014.852, 339569...",18526.246839,751.029470,0.040539,6,JACANA-TREMEMBE
13,35503080925,Manuel Antônio Portella,35,São Paulo,SP,3550308,São Paulo,307,307,307,...,0,0,0,0,"POLYGON ((320297.363 7396761.169, 320304.738 7...",62199.018628,5149.310696,0.082788,25,LAPA
14,35503081739,Fazenda da Juta III,35,São Paulo,SP,3550308,São Paulo,287,287,287,...,0,0,0,0,"MULTIPOLYGON (((347748.38 7386558.489, 347727....",27619.298504,821.096085,0.029729,9,SAPOPEMBA
19,35503080180,Liviero,35,São Paulo,SP,3550308,São Paulo,171,171,171,...,0,0,0,0,"POLYGON ((336911.453 7383279.915, 336906.821 7...",13069.809113,473.095572,0.036198,6,IPIRANGA
20,35503081859,Estrada de Poá,35,São Paulo,SP,3550308,São Paulo,42,42,42,...,0,0,0,0,"MULTIPOLYGON (((357877.212 7394414.473, 357874...",12915.034452,2454.411373,0.190043,8,GUAIANASES
21,35503080965,Francisco Capara,35,São Paulo,SP,3550308,São Paulo,165,165,165,...,0,0,0,0,"POLYGON ((358330.389 7395329.13, 358327.305 73...",17372.458232,5309.107608,0.305605,50,GUAIANASES
22,35503081959,Rua Professor Cosme Deodato Tadeu,35,São Paulo,SP,3550308,São Paulo,65,65,65,...,0,0,0,0,"POLYGON ((357612.105 7394799.908, 357615.945 7...",10215.616185,1424.359596,0.139430,9,GUAIANASES


Como algumas áreas de favelas estão em mais de uma subprefeitura, precisamos remover uma das duplicatas. Vamos manter a primeira ocorrência.

In [52]:
df_favelas_fora = df_favelas_fora.drop_duplicates(subset=['cd_fcu']).reset_index(drop=True)

df_favelas_fora

,cd_fcu,nm_fcu,cd_uf,nm_uf,sigla_uf,cd_mun,nm_mun,Total,Particular,Particular permanente,...,Particular improvisado,Coletivo,Coletivo - com morador,Coletivo - sem morador,geometry,area_total,area_fora,peso,domicilios_favela_fora,nm_subprefeitura
0,35503082104,Rua Antônio Rodrigues da Silva Júnior,35,São Paulo,SP,3550308,São Paulo,77,77,77,...,0,0,0,0,"POLYGON ((357271.721 7388636.046, 357274.522 7...",20903.876963,990.536419,0.047385,4,CIDADE TIRADENTES
1,35503081511,Jardim Divino,35,São Paulo,SP,3550308,São Paulo,73,73,73,...,0,0,0,0,"POLYGON ((357872.964 7394399.925, 357872.094 7...",11321.131655,1465.397866,0.129439,9,GUAIANASES
2,35503080940,Morada do Sol II,35,São Paulo,SP,3550308,São Paulo,1035,1035,1035,...,0,0,0,0,"MULTIPOLYGON (((318091.565 7388515.223, 318060...",61893.612468,4221.421108,0.068204,71,BUTANTA
3,35503081196,São José,35,São Paulo,SP,3550308,São Paulo,143,143,143,...,0,0,0,0,"MULTIPOLYGON (((339568.383 7405014.852, 339569...",18526.246839,751.029470,0.040539,6,JACANA-TREMEMBE
4,35503080925,Manuel Antônio Portella,35,São Paulo,SP,3550308,São Paulo,307,307,307,...,0,0,0,0,"POLYGON ((320297.363 7396761.169, 320304.738 7...",62199.018628,5149.310696,0.082788,25,LAPA
5,35503081739,Fazenda da Juta III,35,São Paulo,SP,3550308,São Paulo,287,287,287,...,0,0,0,0,"MULTIPOLYGON (((347748.38 7386558.489, 347727....",27619.298504,821.096085,0.029729,9,SAPOPEMBA
6,35503080180,Liviero,35,São Paulo,SP,3550308,São Paulo,171,171,171,...,0,0,0,0,"POLYGON ((336911.453 7383279.915, 336906.821 7...",13069.809113,473.095572,0.036198,6,IPIRANGA
7,35503081859,Estrada de Poá,35,São Paulo,SP,3550308,São Paulo,42,42,42,...,0,0,0,0,"MULTIPOLYGON (((357877.212 7394414.473, 357874...",12915.034452,2454.411373,0.190043,8,GUAIANASES
8,35503080965,Francisco Capara,35,São Paulo,SP,3550308,São Paulo,165,165,165,...,0,0,0,0,"POLYGON ((358330.389 7395329.13, 358327.305 73...",17372.458232,5309.107608,0.305605,50,GUAIANASES
9,35503081959,Rua Professor Cosme Deodato Tadeu,35,São Paulo,SP,3550308,São Paulo,65,65,65,...,0,0,0,0,"POLYGON ((357612.105 7394799.908, 357615.945 7...",10215.616185,1424.359596,0.139430,9,GUAIANASES


Agora, podemos agregar o número de domicílios em favelas por subprefeitura e adicionar ao dataframe anterior.

In [53]:
df_fav_sub_fora = (
    df_favelas_fora
    .groupby('nm_subprefeitura', as_index=False)
    .agg(
        domicilios_favela_fora = ('domicilios_favela_fora', 'sum')
    )
)

df_fav_sub_fora

,nm_subprefeitura,domicilios_favela_fora
0,BUTANTA,134
1,CAMPO LIMPO,41
2,CIDADE ADEMAR,62
3,CIDADE TIRADENTES,70
4,GUAIANASES,144
5,IPIRANGA,34
6,ITAIM PAULISTA,24
7,JACANA-TREMEMBE,7
8,LAPA,25
9,M BOI MIRIM,25


### Agregando os domicílios em favelas por subprefeitura

In [54]:
subs_fora = df_fav_sub['nm_subprefeitura'].isin(df_fav_sub_fora['nm_subprefeitura'])

df_fav_sub.loc[subs_fora, 'domicilios_particulares_permanentes_favela'] = (
    df_fav_sub
    .loc[subs_fora]
    .apply(lambda row: row['domicilios_particulares_permanentes_favela']
           + df_fav_sub_fora.loc[df_fav_sub_fora['nm_subprefeitura']==row['nm_subprefeitura'], 'domicilios_favela_fora'].values[0], axis=1)
)

df_fav_sub

,id,cd_identificador_subprefeitura,cd_subprefeitura,nm_subprefeitura,tx_escala,sg_fonte_original,dt_criacao,cd_tipo_discrepancia,dt_atualizacao,cd_usuario_atualizacao,sg_subprefeitura,qt_area_quilometro,qt_area_metro,geometry,domicilios_particulares_permanentes_favela
0,subprefeitura.1,1,02,PIRITUBA-JARAGUA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.042000+00:00,,PJ,55,5.502102e+07,"POLYGON ((318663.925 7404127.712, 318663.251 7...",26730
1,subprefeitura.2,2,03,FREGUESIA-BRASILANDIA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.050000+00:00,,FO,32,3.198020e+07,"POLYGON ((327340.628 7399133.313, 327331.514 7...",32156
2,subprefeitura.3,3,04,CASA VERDE-CACHOEIRINHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.956000+00:00,,CV,27,2.723234e+07,"POLYGON ((329084.795 7402363.669, 329086.123 7...",11697
3,subprefeitura.4,4,05,SANTANA-TUCURUVI,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.068000+00:00,,ST,36,3.578252e+07,"POLYGON ((334076.366 7398045.594, 334074.986 7...",759
4,subprefeitura.5,5,06,JACANA-TREMEMBE,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.950000+00:00,,JT,65,6.511566e+07,"POLYGON ((335167.648 7404409.048, 335167.247 7...",31509
5,subprefeitura.6,6,07,VILA MARIA-VILA GUILHERME,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.073000+00:00,,MG,27,2.689922e+07,"POLYGON ((336762.078 7401144.267, 336794.867 7...",9078
6,subprefeitura.7,7,21,PENHA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.055000+00:00,,PE,40,4.042924e+07,"POLYGON ((346801.065 7402842.892, 346800.103 7...",14156
7,subprefeitura.8,8,22,ERMELINO MATARAZZO,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:42.081000+00:00,,EM,16,1.596639e+07,"POLYGON ((349116.316 7399473.221, 349133.905 7...",9023
8,subprefeitura.9,9,23,SAO MIGUEL,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.944000+00:00,,MP,26,2.615923e+07,"POLYGON ((352480.323 7397515.871, 352477.052 7...",27012
9,subprefeitura.10,10,24,ITAIM PAULISTA,1:5000,GEOGSG,2023-11-30,100200300,2025-07-05 04:29:41.967000+00:00,,IT,22,2.160382e+07,"POLYGON ((356423.064 7397223.44, 356418.471 73...",11274


In [55]:
df_fav_sub['domicilios_particulares_permanentes_favela'].sum()

np.int64(649761)

In [56]:
df_favelas['Particular permanente'].sum()

np.int64(649765)

In [57]:
df_favelas['Particular permanente'].sum()-df_fav_sub['domicilios_particulares_permanentes_favela'].sum()

np.int64(4)

In [58]:
1-df_fav_sub['domicilios_particulares_permanentes_favela'].sum()/df_favelas['Particular permanente'].sum()

np.float64(6.1560718105369006e-06)

Agora sim, temos uma perda de apenas 4 domicílios em favelas, dentro do universo de 649.765 domicílios em favelas, o que representa uma perda de apenas 0,0006%. Ademais, essa perda provavelmente se deve a falhas de arredondamento, então podemos considerar que a estimativa está satisfatória.

### Padronizando os nomes de subprefeituras

In [59]:
subs_favelas = df_fav_sub['nm_subprefeitura'].apply(unidecode).unique().tolist()
subs_favelas.sort()
subs_favelas

['ARICANDUVA-FORMOSA-CARRAO',
 'BUTANTA',
 'CAMPO LIMPO',
 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR',
 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA',
 'GUAIANASES',
 'IPIRANGA',
 'ITAIM PAULISTA',
 'ITAQUERA',
 'JABAQUARA',
 'JACANA-TREMEMBE',
 'LAPA',
 'M BOI MIRIM',
 'MOOCA',
 'PARELHEIROS',
 'PENHA',
 'PERUS',
 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI',
 'SANTO AMARO',
 'SAO MATEUS',
 'SAO MIGUEL',
 'SAPOPEMBA',
 'SE',
 'VILA MARIA-VILA GUILHERME',
 'VILA MARIANA',
 'VILA PRUDENTE']

In [60]:
len(subs_favelas)

31

O tamanho da lista de subs indica que uma das subprefeituras não possui nenhum domicílio em favelas. Inspecionando a lista, vemos que a subprefeitura faltante é a de Pinheiros, o que parece fazer sentido. Primeiro, vamos criar uma cópia da lista de subs do qlik adaptada aos dados de domicílios em favelas.

In [61]:
subs_qlik_favelas = subs_qlik.copy()
subs_qlik_favelas.remove('PINHEIROS')
subs_qlik_favelas

['ARICANDUVA-FORMOSA-CARRAO',
 'BUTANTA',
 'CAMPO LIMPO',
 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR',
 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA',
 'GUAIANASES',
 'IPIRANGA',
 'ITAIM PAULISTA',
 'ITAQUERA',
 'JABAQUARA',
 'JACANA-TREMEMBE',
 'LAPA',
 'M BOI MIRIM',
 'MOOCA',
 'PARELHEIROS',
 'PENHA',
 'PERUS',
 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI',
 'SANTO AMARO',
 'SAO MATEUS',
 'SAO MIGUEL',
 'SAPOPEMBA',
 'SE',
 'VILA MARIA-VILA GUILHERME',
 'VILA MARIANA',
 'VILA PRUDENTE']

In [62]:
mapper_favelas = {
    s: q for s, q in zip(subs_favelas, subs_qlik_favelas)
}

mapper_favelas

{'ARICANDUVA-FORMOSA-CARRAO': 'ARICANDUVA-FORMOSA-CARRAO',
 'BUTANTA': 'BUTANTA',
 'CAMPO LIMPO': 'CAMPO LIMPO',
 'CAPELA DO SOCORRO': 'CAPELA DO SOCORRO',
 'CASA VERDE-CACHOEIRINHA': 'CASA VERDE-CACHOEIRINHA',
 'CIDADE ADEMAR': 'CIDADE ADEMAR',
 'CIDADE TIRADENTES': 'CIDADE TIRADENTES',
 'ERMELINO MATARAZZO': 'ERMELINO MATARAZZO',
 'FREGUESIA-BRASILANDIA': 'FREGUESIA-BRASILANDIA',
 'GUAIANASES': 'GUAIANASES',
 'IPIRANGA': 'IPIRANGA',
 'ITAIM PAULISTA': 'ITAIM PAULISTA',
 'ITAQUERA': 'ITAQUERA',
 'JABAQUARA': 'JABAQUARA',
 'JACANA-TREMEMBE': 'JACANA-TREMEMBE',
 'LAPA': 'LAPA',
 'M BOI MIRIM': 'M BOI MIRIM',
 'MOOCA': 'MOOCA',
 'PARELHEIROS': 'PARELHEIROS',
 'PENHA': 'PENHA',
 'PERUS': 'PERUS',
 'PIRITUBA-JARAGUA': 'PIRITUBA-JARAGUA',
 'SANTANA-TUCURUVI': 'SANTANA-TUCURUVI',
 'SANTO AMARO': 'SANTO AMARO',
 'SAO MATEUS': 'SAO MATEUS',
 'SAO MIGUEL': 'SAO MIGUEL',
 'SAPOPEMBA': 'SAPOPEMBA',
 'SE': 'SE',
 'VILA MARIA-VILA GUILHERME': 'VILA MARIA-VILA GUILHERME',
 'VILA MARIANA': 'VILA MARI

In [63]:
any(s != q for s, q in mapper_favelas.items())

False

### Recriando com base no df_subs

In [64]:
df_fav_sub_final = (
    df_subs
    .merge(df_fav_sub[['nm_subprefeitura',
                          'sg_subprefeitura',
                          'domicilios_particulares_permanentes_favela']],
                how='left',
                left_on='sub.NOME',
                right_on='nm_subprefeitura')
    .drop(columns=['nm_subprefeitura'])
)

df_fav_sub_final


,sub.CODIGO,sub.NOME,sg_subprefeitura,domicilios_particulares_permanentes_favela
0,1,PERUS,PR,14857.0
1,2,PIRITUBA-JARAGUA,PJ,26730.0
2,3,FREGUESIA-BRASILANDIA,FO,32156.0
3,4,CASA VERDE-CACHOEIRINHA,CV,11697.0
4,5,SANTANA-TUCURUVI,ST,759.0
5,6,JACANA-TREMEMBE,JT,31509.0
6,7,VILA MARIA-VILA GUILHERME,MG,9078.0
7,8,LAPA,LA,7047.0
8,9,SE,SE,731.0
9,10,BUTANTA,BT,27623.0


Finalmente, completamos os campos vazios da subprefeitura de Pinheiros.

In [65]:
df_fav_sub_final['domicilios_particulares_permanentes_favela'] = (
    df_fav_sub_final['domicilios_particulares_permanentes_favela']
    .fillna(0)
    .astype(int)
)

df_fav_sub_final.loc[df_fav_sub_final['sub.NOME']=='PINHEIROS', 'sg_subprefeitura'] = (
    gdf_subs.loc[gdf_subs['nm_subprefeitura']=='PINHEIROS', 'sg_subprefeitura'].values[0]
)

df_fav_sub_final

,sub.CODIGO,sub.NOME,sg_subprefeitura,domicilios_particulares_permanentes_favela
0,1,PERUS,PR,14857
1,2,PIRITUBA-JARAGUA,PJ,26730
2,3,FREGUESIA-BRASILANDIA,FO,32156
3,4,CASA VERDE-CACHOEIRINHA,CV,11697
4,5,SANTANA-TUCURUVI,ST,759
5,6,JACANA-TREMEMBE,JT,31509
6,7,VILA MARIA-VILA GUILHERME,MG,9078
7,8,LAPA,LA,7047
8,9,SE,SE,731
9,10,BUTANTA,BT,27623


## Orçamento do Programa de habitação

Para o orçamento, além de padronizar os nomes de subprefeituras e tipos de dados das métricas, precisaremos também adaptar os dados para compatibilizar o orçamento regionalizado e não realizado. Para isso, vamos fazer o seguinte:

1. Classificar o orçamento detalhado por nível de regionalização nas seguintes categorias: subprefeitura, região e não regionalizável;
1. Agrupar o restante do orçamento não detalhado e manter apenas o orçamento inicial, atualizado e liquidado;
1. Subtrair o total do orçamento detalhado do orçamento não detalhado e classificar o nível de regionalização como não regionalizado;
1. Unir os dois dataframes de orçamento de acordo com as dimensões mantidas.

### Orçamento regionalizado

In [66]:
df_orcamento_r.head(1)

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,DESCRIÇÃO_FONTE,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,01,41426,2024,94288,2024,2024-04-05 00:00:00.0000000,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,1,759,1224,0,Oeste,Subprefeitura Butantã,Supra-Distrital,Despesa Regionalizável,531084.11


In [67]:
cols_orcamento_r = ['CÓDIGO_ÓRGÃO', 'SIGLA_ÓRGÃO', 'DESCRIÇÃO_ÓRGÃO',
                    'CÓDIGO_PROGRAMA', 'DESCRIÇÃO_PROGRAMA',
                    'CÓDIGO_PROJ_ATIV', 'DESCRIÇÃO_PROJ_ATIV',
                    'CÓDIGO_VÍNCULO_PMSP', 'REGIÃO',
                    'SUBPREFEITURA', 'TIPO_REGIONALIZAÇÃO']

cols_orcamento_r_vl = ['VALOR_DETALHAMENTO_AÇÃO']

df_orcamento_r = df_orcamento_r[cols_orcamento_r + cols_orcamento_r_vl]
df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,531084.11
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,231882.59
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,240538.93
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,294777.93
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,299606.31
...,...,...,...,...,...,...,...,...,...,...,...,...
6615,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,940786.45
6616,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1407083.17
6617,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1936628.40
6618,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,815880.37


In [68]:
df_orcamento_r['TIPO_REGIONALIZAÇÃO'].value_counts()

TIPO_REGIONALIZAÇÃO
Despesa Regionalizável        3069
Despesa Não-Regionalizável    2118
Name: count, dtype: int64

In [69]:
df_orcamento_r.loc[df_orcamento_r['TIPO_REGIONALIZAÇÃO'].isna(), 'TIPO_REGIONALIZAÇÃO'] = 'Despesa Não-Regionalizável'
df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,531084.11
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,231882.59
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,240538.93
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,294777.93
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,299606.31
...,...,...,...,...,...,...,...,...,...,...,...,...
6615,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,940786.45
6616,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1407083.17
6617,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1936628.40
6618,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,815880.37


In [70]:
df_orcamento_r['TIPO_REGIONALIZAÇÃO'].value_counts()

TIPO_REGIONALIZAÇÃO
Despesa Não-Regionalizável    3551
Despesa Regionalizável        3069
Name: count, dtype: int64

In [71]:
df_orcamento_r = df_orcamento_r.groupby(cols_orcamento_r).sum().round(2).reset_index()

df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5.650217e+06
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,5.781284e+05
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,1.422529e+07
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,3.920524e+04
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1.177145e+06
...,...,...,...,...,...,...,...,...,...,...,...,...
232,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Capela do Socorro,Despesa Regionalizável,2.131483e+08
233,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Cidade Ademar,Despesa Regionalizável,7.339787e+07
234,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,4.544281e+07
235,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Parelheiros,Despesa Regionalizável,7.700302e+07


In [72]:
df_orcamento_r.loc[
    ~df_orcamento_r['REGIÃO'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Região'

df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,NIVEL_REGIONALIZAÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5.650217e+06,Região
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,5.781284e+05,Região
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,1.422529e+07,Região
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,3.920524e+04,Região
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1.177145e+06,Região
...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Capela do Socorro,Despesa Regionalizável,2.131483e+08,Região
233,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Cidade Ademar,Despesa Regionalizável,7.339787e+07,Região
234,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,4.544281e+07,Região
235,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Parelheiros,Despesa Regionalizável,7.700302e+07,Região


In [73]:
df_orcamento_r.loc[
    ~df_orcamento_r['SUBPREFEITURA'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Subprefeitura'

df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,NIVEL_REGIONALIZAÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5.650217e+06,Subprefeitura
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,5.781284e+05,Subprefeitura
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,1.422529e+07,Subprefeitura
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,3.920524e+04,Subprefeitura
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1.177145e+06,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Capela do Socorro,Despesa Regionalizável,2.131483e+08,Subprefeitura
233,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Cidade Ademar,Despesa Regionalizável,7.339787e+07,Subprefeitura
234,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,4.544281e+07,Subprefeitura
235,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Parelheiros,Despesa Regionalizável,7.700302e+07,Subprefeitura


In [74]:
df_orcamento_r.loc[df_orcamento_r['NIVEL_REGIONALIZAÇÃO'].isna(), 'NIVEL_REGIONALIZAÇÃO'] = 'Não regionalizável'
df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,NIVEL_REGIONALIZAÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5.650217e+06,Subprefeitura
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,5.781284e+05,Subprefeitura
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,1.422529e+07,Subprefeitura
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,3.920524e+04,Subprefeitura
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1.177145e+06,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Capela do Socorro,Despesa Regionalizável,2.131483e+08,Subprefeitura
233,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Cidade Ademar,Despesa Regionalizável,7.339787e+07,Subprefeitura
234,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,4.544281e+07,Subprefeitura
235,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Parelheiros,Despesa Regionalizável,7.700302e+07,Subprefeitura


### Orçamento não regionalizado

In [75]:
df_orcamento.head(1)

,DataInicial,DataFinal,Cd_AnoExecucao,Cd_Exercicio,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_Congelado,Vl_Descongelado,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao
0,01/01/2024,31/12/2024,2024,2024,169113,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0.0,0.0,0.0,1000,0.0,0.0,0.0,0.0,1000,2025-01-18


In [76]:
cols_orcamento = ['Cd_Orgao', 'Sigla_Orgao', 'Ds_Orgao', 'Cd_Programa',
                  'Ds_Programa', 'ProjetoAtividade', 'Ds_Projeto_Atividade',
                  'COD_VINC_REC_PMSP']

cols_orcamento_vl = ['Vl_Orcado_Ano', 'Vl_Orcado_Atualizado', 'Vl_Liquidado']

df_orcamento_original = df_orcamento.copy()
df_orcamento = df_orcamento[cols_orcamento + cols_orcamento_vl]
df_orcamento

,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Programa,Ds_Programa,ProjetoAtividade,Ds_Projeto_Atividade,COD_VINC_REC_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,1224,1000.0,1000.0,0.00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,1224,1000.0,1000.0,0.00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,1224,1000.0,1000.0,0.00
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,34985177.0,14985177.0,14842626.58
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,0.0,10000000.0,6827361.22
...,...,...,...,...,...,...,...,...,...,...,...
391,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3358,Locação Social,9001,0.0,0.0,0.00
392,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3358,Locação Social,8011,0.0,0.0,0.00
393,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,4353,Manutenção de Unidades Habitacionais,8011,14545000.0,14545000.0,5565823.14
394,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,4353,Manutenção de Unidades Habitacionais,9001,0.0,0.0,0.00


In [77]:
df_orcamento = df_orcamento.groupby(cols_orcamento).sum().reset_index()
df_orcamento

,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Programa,Ds_Programa,ProjetoAtividade,Ds_Projeto_Atividade,COD_VINC_REC_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,1224,1000.0,1.000000e+03,0.000000e+00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,1224,1000.0,1.000000e+03,0.000000e+00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,44985177.0,2.498518e+07,2.166999e+07
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,1224,1000.0,1.000000e+03,0.000000e+00
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,1224,1000.0,1.000000e+03,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,3000.0,3.160488e+07,1.895520e+07
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,3000.0,3.000000e+03,0.000000e+00
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,51895443.0,2.029357e+07,1.991510e+07
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,1000.0,1.000000e+03,0.000000e+00


In [78]:
r_agg_cols = ['CÓDIGO_ÓRGÃO', 'CÓDIGO_PROGRAMA', 'CÓDIGO_PROJ_ATIV',
              'CÓDIGO_VÍNCULO_PMSP']

agg_cols = ['Cd_Orgao', 'Cd_Programa', 'ProjetoAtividade',
            'COD_VINC_REC_PMSP']

df_orcamento_r_agg = (
    df_orcamento_r[r_agg_cols + ['VALOR_DETALHAMENTO_AÇÃO']]
    .groupby(r_agg_cols)
    .sum()
    .reset_index()
)

df_orcamento_r_agg.loc[:, 'VALOR_DETALHAMENTO_AÇÃO'] = (
    df_orcamento_r_agg
    .loc[:, 'VALOR_DETALHAMENTO_AÇÃO']
    .round(2)
)

df_orcamento_ajustado = df_orcamento.merge(
    df_orcamento_r_agg,
    left_on=agg_cols,
    right_on=r_agg_cols,
    how='left'
).drop(columns=r_agg_cols)

df_orcamento_ajustado

,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Programa,Ds_Programa,ProjetoAtividade,Ds_Projeto_Atividade,COD_VINC_REC_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,1224,1000.0,1.000000e+03,0.000000e+00,NaN
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,1224,1000.0,1.000000e+03,0.000000e+00,NaN
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,44985177.0,2.498518e+07,2.166999e+07,2.166999e+07
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,1224,1000.0,1.000000e+03,0.000000e+00,NaN
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,1224,1000.0,1.000000e+03,0.000000e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,3000.0,3.160488e+07,1.895520e+07,1.876643e+07
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,3000.0,3.000000e+03,0.000000e+00,NaN
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,51895443.0,2.029357e+07,1.991510e+07,1.991510e+07
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,1000.0,1.000000e+03,0.000000e+00,NaN


In [79]:
df_orcamento_ajustado.loc[df_orcamento_ajustado['VALOR_DETALHAMENTO_AÇÃO'].isna(), 'VALOR_DETALHAMENTO_AÇÃO'] = 0

df_orcamento_ajustado.loc[:, 'Vl_Liquidado_N_Detalhado'] = (
    df_orcamento_ajustado.loc[:, 'Vl_Liquidado']
    - df_orcamento_ajustado.loc[:, 'VALOR_DETALHAMENTO_AÇÃO']).round(2)

df_orcamento_ajustado

,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Programa,Ds_Programa,ProjetoAtividade,Ds_Projeto_Atividade,COD_VINC_REC_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO,Vl_Liquidado_N_Detalhado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,1224,1000.0,1.000000e+03,0.000000e+00,0.000000e+00,0.00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,1224,1000.0,1.000000e+03,0.000000e+00,0.000000e+00,0.00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,44985177.0,2.498518e+07,2.166999e+07,2.166999e+07,0.00
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,1224,1000.0,1.000000e+03,0.000000e+00,0.000000e+00,0.00
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,1224,1000.0,1.000000e+03,0.000000e+00,0.000000e+00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,3000.0,3.160488e+07,1.895520e+07,1.876643e+07,188764.39
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,3000.0,3.000000e+03,0.000000e+00,0.000000e+00,0.00
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,51895443.0,2.029357e+07,1.991510e+07,1.991510e+07,0.00
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,1000.0,1.000000e+03,0.000000e+00,0.000000e+00,0.00


In [80]:
df_orcamento_ajustado[df_orcamento_ajustado['Vl_Liquidado_N_Detalhado']<0]

,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Programa,Ds_Programa,ProjetoAtividade,Ds_Projeto_Atividade,COD_VINC_REC_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO,Vl_Liquidado_N_Detalhado


In [81]:
df_orcamento_ajustado[['Vl_Liquidado', 'VALOR_DETALHAMENTO_AÇÃO']].sum()

Vl_Liquidado               4.914679e+09
VALOR_DETALHAMENTO_AÇÃO    4.682109e+09
dtype: float64

### Unindo os dados de orçamento

Agora, vamos adicionar os dados não detalhados ao dataframe que contém o orçamento detalhado.

In [82]:
orcamento_cols_map = {'Cd_Orgao': 'CÓDIGO_ÓRGÃO',
                      'Sigla_Orgao': 'SIGLA_ÓRGÃO',
                      'Ds_Orgao': 'DESCRIÇÃO_ÓRGÃO',
                      'Cd_Programa': 'CÓDIGO_PROGRAMA',
                      'Ds_Programa': 'DESCRIÇÃO_PROGRAMA',
                      'ProjetoAtividade': 'CÓDIGO_PROJ_ATIV',
                      'Ds_Projeto_Atividade': 'DESCRIÇÃO_PROJ_ATIV',
                      'COD_VINC_REC_PMSP': 'CÓDIGO_VÍNCULO_PMSP',
                      'Vl_Liquidado_N_Detalhado': 'Vl_Liquidado'}

df_orcamento_ajustado = (
    df_orcamento_ajustado
    .drop(columns=['VALOR_DETALHAMENTO_AÇÃO', 'Vl_Liquidado'])
    .rename(columns=orcamento_cols_map)
    )

df_orcamento_ajustado

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,Vl_Orcado_Ano,Vl_Orcado_Atualizado,Vl_Liquidado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,1224,1000.0,1.000000e+03,0.00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,1224,1000.0,1.000000e+03,0.00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,44985177.0,2.498518e+07,0.00
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,1224,1000.0,1.000000e+03,0.00
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,1224,1000.0,1.000000e+03,0.00
...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,3000.0,3.160488e+07,188764.39
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,3000.0,3.000000e+03,0.00
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,51895443.0,2.029357e+07,0.00
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,1000.0,1.000000e+03,0.00


In [83]:
df_orcamento_r = (df_orcamento_r
                  .rename(columns={'VALOR_DETALHAMENTO_AÇÃO': 'Vl_Liquidado'}))

df_orcamento_r

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5.650217e+06,Subprefeitura
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,5.781284e+05,Subprefeitura
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,1.422529e+07,Subprefeitura
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,3.920524e+04,Subprefeitura
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1.177145e+06,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Capela do Socorro,Despesa Regionalizável,2.131483e+08,Subprefeitura
233,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Cidade Ademar,Despesa Regionalizável,7.339787e+07,Subprefeitura
234,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,4.544281e+07,Subprefeitura
235,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,0402,Sul,Subprefeitura Parelheiros,Despesa Regionalizável,7.700302e+07,Subprefeitura


In [84]:
df_orcamento_final = pd.concat([df_orcamento_r, df_orcamento_ajustado])

df_orcamento_final

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5650216.54,Subprefeitura,NaN,NaN
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,578128.38,Subprefeitura,NaN,NaN
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,14225292.96,Subprefeitura,NaN,NaN
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,39205.24,Subprefeitura,NaN,NaN
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1177144.68,Subprefeitura,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,NaN,NaN,NaN,188764.39,NaN,3000.0,3.160488e+07
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,NaN,NaN,NaN,0.00,NaN,3000.0,3.000000e+03
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,NaN,NaN,NaN,0.00,NaN,51895443.0,2.029357e+07
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,NaN,NaN,NaN,0.00,NaN,1000.0,1.000000e+03


In [85]:
df_orcamento_final.loc[df_orcamento_final['Vl_Orcado_Ano'].isna(),
                       'Vl_Orcado_Ano'] = 0
df_orcamento_final.loc[df_orcamento_final['Vl_Orcado_Atualizado'].isna(),
                       'Vl_Orcado_Atualizado'] = 0
df_orcamento_final.loc[df_orcamento_final['NIVEL_REGIONALIZAÇÃO'].isna(),
                       'NIVEL_REGIONALIZAÇÃO'] = 'Não detalhado'

df_orcamento_final

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5650216.54,Subprefeitura,0.0,0.000000e+00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,578128.38,Subprefeitura,0.0,0.000000e+00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,14225292.96,Subprefeitura,0.0,0.000000e+00
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,39205.24,Subprefeitura,0.0,0.000000e+00
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1177144.68,Subprefeitura,0.0,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,0402,NaN,NaN,NaN,188764.39,Não detalhado,3000.0,3.160488e+07
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,NaN,NaN,NaN,0.00,Não detalhado,3000.0,3.000000e+03
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,0402,NaN,NaN,NaN,0.00,Não detalhado,51895443.0,2.029357e+07
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,NaN,NaN,NaN,0.00,Não detalhado,1000.0,1.000000e+03


### Padronizando os nomes de subprefeituras

In [86]:
subs_orcamento = (
    df_orcamento_final.loc[~df_orcamento_final['SUBPREFEITURA'].isna(), 'SUBPREFEITURA']
    .apply(unidecode)
    .unique()
    .tolist()
)

subs_orcamento.sort()

subs_orcamento

['Subprefeitura Aricanduva/Formosa/Carrao',
 'Subprefeitura Butanta',
 'Subprefeitura Campo Limpo',
 'Subprefeitura Capela do Socorro',
 'Subprefeitura Casa Verde/Cachoeirinha',
 'Subprefeitura Cidade Ademar',
 'Subprefeitura Cidade Tiradentes',
 'Subprefeitura Freguesia/Brasilandia',
 'Subprefeitura Ipiranga',
 'Subprefeitura Itaim Paulista',
 'Subprefeitura Itaquera',
 'Subprefeitura Jabaquara',
 'Subprefeitura Jacana/Tremembe',
 'Subprefeitura Lapa',
 "Subprefeitura M'Boi Mirim",
 'Subprefeitura Mooca',
 'Subprefeitura Parelheiros',
 'Subprefeitura Penha',
 'Subprefeitura Perus/Anhanguera',
 'Subprefeitura Pinheiros',
 'Subprefeitura Santana/Tucuruvi',
 'Subprefeitura Santo Amaro',
 'Subprefeitura Sao Mateus',
 'Subprefeitura Sao Miguel Paulista',
 'Subprefeitura Sapopemba',
 'Subprefeitura Se',
 'Subprefeitura Vila Maria/Vila Guilherme',
 'Subprefeitura de Guaianases',
 'Subprefeitura de Vila Prudente',
 'Supra Subprefeitura',
 'Supra Subprefeitura Centro',
 'Supra Subprefeitura Le

In [87]:
subs_orcamento[:-6]

['Subprefeitura Aricanduva/Formosa/Carrao',
 'Subprefeitura Butanta',
 'Subprefeitura Campo Limpo',
 'Subprefeitura Capela do Socorro',
 'Subprefeitura Casa Verde/Cachoeirinha',
 'Subprefeitura Cidade Ademar',
 'Subprefeitura Cidade Tiradentes',
 'Subprefeitura Freguesia/Brasilandia',
 'Subprefeitura Ipiranga',
 'Subprefeitura Itaim Paulista',
 'Subprefeitura Itaquera',
 'Subprefeitura Jabaquara',
 'Subprefeitura Jacana/Tremembe',
 'Subprefeitura Lapa',
 "Subprefeitura M'Boi Mirim",
 'Subprefeitura Mooca',
 'Subprefeitura Parelheiros',
 'Subprefeitura Penha',
 'Subprefeitura Perus/Anhanguera',
 'Subprefeitura Pinheiros',
 'Subprefeitura Santana/Tucuruvi',
 'Subprefeitura Santo Amaro',
 'Subprefeitura Sao Mateus',
 'Subprefeitura Sao Miguel Paulista',
 'Subprefeitura Sapopemba',
 'Subprefeitura Se',
 'Subprefeitura Vila Maria/Vila Guilherme',
 'Subprefeitura de Guaianases',
 'Subprefeitura de Vila Prudente']

In [88]:
len(subs_orcamento[:-6])

29

In [89]:
subs_qlik_orcamento = subs_qlik.copy()
# As 3 subs abaixo não aparecem na lista de liquidação do orçamento
subs_qlik_orcamento.remove('ERMELINO MATARAZZO')
subs_qlik_orcamento.remove('PIRITUBA-JARAGUA')
subs_qlik_orcamento.remove('VILA MARIANA')
# Guainases e Vila Prudente aparecem em uma ordenação diferente, por isso serão
# removidas e adicionadas novamente ao final da lista
subs_qlik_orcamento.remove('GUAIANASES')
subs_qlik_orcamento.remove('VILA PRUDENTE')
subs_qlik_orcamento.append('GUAIANASES')
subs_qlik_orcamento.append('VILA PRUDENTE')

In [90]:
mapper_orcamento = {
    so: sq
    for so, sq in zip(subs_orcamento, subs_qlik_orcamento)
}

mapper_orcamento

{'Subprefeitura Aricanduva/Formosa/Carrao': 'ARICANDUVA-FORMOSA-CARRAO',
 'Subprefeitura Butanta': 'BUTANTA',
 'Subprefeitura Campo Limpo': 'CAMPO LIMPO',
 'Subprefeitura Capela do Socorro': 'CAPELA DO SOCORRO',
 'Subprefeitura Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
 'Subprefeitura Cidade Ademar': 'CIDADE ADEMAR',
 'Subprefeitura Cidade Tiradentes': 'CIDADE TIRADENTES',
 'Subprefeitura Freguesia/Brasilandia': 'FREGUESIA-BRASILANDIA',
 'Subprefeitura Ipiranga': 'IPIRANGA',
 'Subprefeitura Itaim Paulista': 'ITAIM PAULISTA',
 'Subprefeitura Itaquera': 'ITAQUERA',
 'Subprefeitura Jabaquara': 'JABAQUARA',
 'Subprefeitura Jacana/Tremembe': 'JACANA-TREMEMBE',
 'Subprefeitura Lapa': 'LAPA',
 "Subprefeitura M'Boi Mirim": 'M BOI MIRIM',
 'Subprefeitura Mooca': 'MOOCA',
 'Subprefeitura Parelheiros': 'PARELHEIROS',
 'Subprefeitura Penha': 'PENHA',
 'Subprefeitura Perus/Anhanguera': 'PERUS',
 'Subprefeitura Pinheiros': 'PINHEIROS',
 'Subprefeitura Santana/Tucuruvi': 'SANTANA-TUCURUVI'

In [91]:
df_orcamento_final.insert(
    7,
    'sub.NOME',
    df_orcamento_final.loc[:,'SUBPREFEITURA'].apply(lambda s: unidecode(s) if isinstance(s, str) else None).map(mapper_orcamento)
)

df_orcamento_final

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,SE,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5650216.54,Subprefeitura,0.0,0.000000e+00
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,CASA VERDE-CACHOEIRINHA,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,578128.38,Subprefeitura,0.0,0.000000e+00
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,BUTANTA,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,14225292.96,Subprefeitura,0.0,0.000000e+00
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,IPIRANGA,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,39205.24,Subprefeitura,0.0,0.000000e+00
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,M BOI MIRIM,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1177144.68,Subprefeitura,0.0,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,NaN,0402,NaN,NaN,NaN,188764.39,Não detalhado,3000.0,3.160488e+07
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,3000.0,3.000000e+03
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,51895443.0,2.029357e+07
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,1000.0,1.000000e+03


In [92]:
df_orcamento_final.loc[:, 'ano'] = 2024

# df_orcamento_final = df_orcamento_final.merge(df_subs_ano,
#                               how='left',
#                               on=['sub.NOME', 'ano'])

df_orcamento_final

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado,ano
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,SE,1224,Centro,Subprefeitura Sé,Despesa Regionalizável,5650216.54,Subprefeitura,0.0,0.000000e+00,2024
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,CASA VERDE-CACHOEIRINHA,1224,Norte,Subprefeitura Casa Verde/Cachoeirinha,Despesa Regionalizável,578128.38,Subprefeitura,0.0,0.000000e+00,2024
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,BUTANTA,1224,Oeste,Subprefeitura Butantã,Despesa Regionalizável,14225292.96,Subprefeitura,0.0,0.000000e+00,2024
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,IPIRANGA,1224,Sul,Subprefeitura Ipiranga,Despesa Regionalizável,39205.24,Subprefeitura,0.0,0.000000e+00,2024
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,M BOI MIRIM,1224,Sul,Subprefeitura M'Boi Mirim,Despesa Regionalizável,1177144.68,Subprefeitura,0.0,0.000000e+00,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3354,Construção de Unidades Habitacionais,NaN,0402,NaN,NaN,NaN,188764.39,Não detalhado,3000.0,3.160488e+07,2024
101,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,3000.0,3.000000e+03,2024
102,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3357,Urbanização de Favelas,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,51895443.0,2.029357e+07,2024
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,NaN,0402,NaN,NaN,NaN,0.00,Não detalhado,1000.0,1.000000e+03,2024


### Adicionando as subprefeituras faltantes

In [93]:
orcamento_subs_cols = ['CÓDIGO_ÓRGÃO', 'SIGLA_ÓRGÃO', 'DESCRIÇÃO_ÓRGÃO',
                       'CÓDIGO_PROGRAMA', 'DESCRIÇÃO_PROGRAMA',
                       'CÓDIGO_PROJ_ATIV', 'DESCRIÇÃO_PROJ_ATIV',
                       'CÓDIGO_VÍNCULO_PMSP', 'ano']

df_orcamento_subs = df_orcamento_final[orcamento_subs_cols].copy()
df_orcamento_subs = df_orcamento_subs.drop_duplicates().reset_index(drop=True)
df_orcamento_subs

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,ano
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024
1,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,1276,Projetos e Ações de Apoio Habitacional,1630,2024
2,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,2635,Serviço de Moradia Transitória,9001,2024
3,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,0000,2024
4,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,0003,2024
...,...,...,...,...,...,...,...,...,...
100,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,9001,2024
101,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3358,Locação Social,9001,2024
102,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,4353,Manutenção de Unidades Habitacionais,9001,2024
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,2024


In [94]:
df_orcamento_subs['TIPO_REGIONALIZAÇÃO'] = 'Despesa Regionalizável'
df_orcamento_subs['NIVEL_REGIONALIZAÇÃO'] = 'Subprefeitura'

df_orcamento_subs

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,ano,TIPO_REGIONALIZAÇÃO,NIVEL_REGIONALIZAÇÃO
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura
1,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,1276,Projetos e Ações de Apoio Habitacional,1630,2024,Despesa Regionalizável,Subprefeitura
2,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,2635,Serviço de Moradia Transitória,9001,2024,Despesa Regionalizável,Subprefeitura
3,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,0000,2024,Despesa Regionalizável,Subprefeitura
4,14,SEHAB,Secretaria Municipal de Habitação,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,0003,2024,Despesa Regionalizável,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...
100,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,9001,2024,Despesa Regionalizável,Subprefeitura
101,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,3358,Locação Social,9001,2024,Despesa Regionalizável,Subprefeitura
102,91,FMH,Fundo Municipal de Habitação,3002,Acesso à Moradia Adequada,4353,Manutenção de Unidades Habitacionais,9001,2024,Despesa Regionalizável,Subprefeitura
103,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,0402,2024,Despesa Regionalizável,Subprefeitura


In [95]:
df_orcamento_subs = (
    df_orcamento_subs
    .merge(pd.DataFrame(columns=['sub.NOME'], data=subs_qlik),
           how='cross')
)

df_orcamento_subs

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,CÓDIGO_VÍNCULO_PMSP,ano,TIPO_REGIONALIZAÇÃO,NIVEL_REGIONALIZAÇÃO,sub.NOME
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura,ARICANDUVA-FORMOSA-CARRAO
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura,BUTANTA
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura,CAMPO LIMPO
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura,CAPELA DO SOCORRO
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3356,Regularização Fundiária,1224,2024,Despesa Regionalizável,Subprefeitura,CASA VERDE-CACHOEIRINHA
...,...,...,...,...,...,...,...,...,...,...,...,...
3355,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,2024,Despesa Regionalizável,Subprefeitura,SAPOPEMBA
3356,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,2024,Despesa Regionalizável,Subprefeitura,SE
3357,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,2024,Despesa Regionalizável,Subprefeitura,VILA MARIA-VILA GUILHERME
3358,98,FUNDURB,Fundo de Desenvolvimento Urbano,3002,Acesso à Moradia Adequada,3358,Locação Social,0402,2024,Despesa Regionalizável,Subprefeitura,VILA MARIANA


In [96]:
df_orcamento_completo = (
    df_orcamento_final
    .merge(df_orcamento_subs,
           how='outer',
           on=df_orcamento_subs.columns.tolist())
)

df_orcamento_completo

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado,ano
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,ARICANDUVA-FORMOSA-CARRAO,1224,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,BUTANTA,1224,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAMPO LIMPO,1224,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAPELA DO SOCORRO,1224,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CASA VERDE-CACHOEIRINHA,1224,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,SE,0402,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
3519,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIA-VILA GUILHERME,0402,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
3520,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIANA,0402,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024
3521,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA PRUDENTE,0402,NaN,NaN,Despesa Regionalizável,NaN,Subprefeitura,NaN,NaN,2024


In [97]:
df_orcamento_completo.loc[df_orcamento_completo['Vl_Liquidado'].isna(), 'Vl_Liquidado'] = 0
df_orcamento_completo.loc[df_orcamento_completo['Vl_Orcado_Ano'].isna(), 'Vl_Orcado_Ano'] = 0
df_orcamento_completo.loc[df_orcamento_completo['Vl_Orcado_Atualizado'].isna(), 'Vl_Orcado_Atualizado'] = 0

df_orcamento_completo

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado,ano
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,ARICANDUVA-FORMOSA-CARRAO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,BUTANTA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAMPO LIMPO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAPELA DO SOCORRO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CASA VERDE-CACHOEIRINHA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,SE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
3519,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIA-VILA GUILHERME,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
3520,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIANA,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024
3521,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA PRUDENTE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024


### Adicionando descrição das vinculações

In [98]:
df_orcamento_completo = (
    df_orcamento_completo
    .merge(df_orcamento_original[['COD_VINC_REC_PMSP', 'TXT_VINC_PMSP']].drop_duplicates(),
            how='left',
            left_on='CÓDIGO_VÍNCULO_PMSP',
            right_on='COD_VINC_REC_PMSP')
    .drop(columns='COD_VINC_REC_PMSP')
)

df_orcamento_completo

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado,ano,TXT_VINC_PMSP
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,ARICANDUVA-FORMOSA-CARRAO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,BUTANTA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAMPO LIMPO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAPELA DO SOCORRO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CASA VERDE-CACHOEIRINHA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,SE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...
3519,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIA-VILA GUILHERME,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...
3520,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIANA,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...
3521,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA PRUDENTE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...


### Adicionando a chave composta subprefeitura-ano

In [99]:
df_orcamento_completo = df_orcamento_completo.merge(df_subs_ano,
                                                    how='left',
                                                    left_on=['sub.NOME', 'ano'],
                                                    right_on=['sub.NOME', 'ano'])

df_orcamento_completo

,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,CÓDIGO_PROJ_ATIV,DESCRIÇÃO_PROJ_ATIV,sub.NOME,CÓDIGO_VÍNCULO_PMSP,REGIÃO,SUBPREFEITURA,TIPO_REGIONALIZAÇÃO,Vl_Liquidado,NIVEL_REGIONALIZAÇÃO,Vl_Orcado_Ano,Vl_Orcado_Atualizado,ano,TXT_VINC_PMSP,subprefeitura-ano
0,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,ARICANDUVA-FORMOSA-CARRAO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,ARICANDUVA-FORMOSA-CARRAO | 2024
1,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,BUTANTA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,BUTANTA | 2024
2,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAMPO LIMPO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,CAMPO LIMPO | 2024
3,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CAPELA DO SOCORRO,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,CAPELA DO SOCORRO | 2024
4,07,FMD,Fundo Municipal de Desenvolvimento Social,3002,Acesso à Moradia Adequada,3340,Programa Pode Entrar,CASA VERDE-CACHOEIRINHA,1224,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SF/COADM-Fundo Municipal de Desenvolvimen...,CASA VERDE-CACHOEIRINHA | 2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3518,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,SE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,SE | 2024
3519,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIA-VILA GUILHERME,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,VILA MARIA-VILA GUILHERME | 2024
3520,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA MARIANA,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,VILA MARIANA | 2024
3521,98,FUNDURB,Fundo de Desenvolvimento Urbano,3005,Promoção da Sustentabilidade Ambiental,3355,Execução do Programa de Mananciais,VILA PRUDENTE,0402,NaN,NaN,Despesa Regionalizável,0.00,Subprefeitura,0.0,0.000000e+00,2024,PMSP-SMDU/ Fundo de Desenvolvimento Urbano-FUN...,VILA PRUDENTE | 2024


# Armazenamento

Finalmente, vamos exportar os dados em formato csv compatível com o Qlik e no padrão do excel para português do Brasil.

In [100]:
base_path = path.join('data_output', 'urbanismo')

if not path.exists(base_path):
    makedirs(base_path)

for name, df in [('orcamento-habitacao', df_orcamento_final),
                 ('producao-his', df_his),
                 ('pdm-meta-12', df_meta_12),
                 ('emissoes-tpu', df_tpu),
                 ('subprefeitura-ano', df_subs_ano),
                 ('domicilios-favela', df_fav_sub_final),]:

    filepath = path.join(base_path, f'{name}.csv')

    df.to_csv(filepath,
              index=False,
              sep=';',
              decimal=',',
              encoding='latin1')